<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Cuadernos/19_LangChain_LangGraph_Agentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">LangChain y LangGraph: aplicaciones y agentes con LLM</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Introducción</span>](#introduccion)
- [<span style="color: #005C99;">Configuración y proveedores</span>](#configuracion-y-proveedores)
- [<span style="color: #005C99;">Modelos de chat</span>](#modelos-de-chat)
- [<span style="color: #005C99;">Plantillas de prompt</span>](#plantillas-de-prompt)
- [<span style="color: #005C99;">Salida estructurada</span>](#salida-estructurada)
- [<span style="color: #005C99;">LCEL: composición con Runnables</span>](#lcel-composicion-con-runnables)
- [<span style="color: #005C99;">Tool calling</span>](#tool-calling)
- [<span style="color: #005C99;">RAG con LangChain</span>](#rag-con-langchain)
- [<span style="color: #005C99;">LangGraph: fundamentos</span>](#langgraph-fundamentos)
- [<span style="color: #005C99;">Workflows con LangGraph</span>](#workflows-con-langgraph)
- [<span style="color: #005C99;">Persistencia, intervención humana y streaming</span>](#persistencia-intervencion-humana-y-streaming)
- [<span style="color: #005C99;">Agentes con LangGraph</span>](#agentes-con-langgraph)
- [<span style="color: #005C99;">Control y seguridad de agentes</span>](#control-y-seguridad-de-agentes)
- [<span style="color: #005C99;">Sistemas multiagente</span>](#sistemas-multiagente)
- [<span style="color: #005C99;">Observabilidad y costos</span>](#observabilidad-y-costos)
- [<span style="color: #005C99;">Cuándo usar estas librerías</span>](#cuando-usar-estas-librerias)
- [<span style="color: #005C99;">Referencias</span>](#referencias)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [1]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib python-dotenv pydantic langchain langchain-openai langchain-google-genai langchain-ollama langchain-text-splitters langgraph grandalf

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import json
import time
import operator
import asyncio
import warnings
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from typing import Annotated, Literal
from typing_extensions import TypedDict

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

SEED = 42
pd.set_option("display.max_colwidth", 120)
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")  # aviso inofensivo de una librería

# Estas librerías cambian con rapidez: anotamos las versiones con las que se ejecutó este cuaderno.
print({p: version(p) for p in ["langchain", "langchain-core", "langgraph", "langchain-openai", "langchain-google-genai", "langchain-ollama"]})

{'langchain': '1.4.7', 'langchain-core': '1.7.3', 'langgraph': '1.2.19', 'langchain-openai': '1.7.3', 'langchain-google-genai': '4.4.2', 'langchain-ollama': '1.1.0'}


<a id="introduccion"></a>
<h2 style="color: #007ACC;">Introducción</h2>

En el cuaderno 18 consultamos los LLM directamente con el SDK de cada proveedor y escribimos a mano una capa común: mensajes con roles, reintentos, paralelismo y salida estructurada. Cuando una aplicación crece (varias llamadas encadenadas, herramientas, documentos, memoria, flujos con decisiones y ciclos), mantener esa capa propia se vuelve costoso. **LangChain** y **LangGraph** son dos librerías de código abierto que ofrecen esa capa ya construida:

- **LangChain** aporta los **componentes**: una interfaz común para los modelos de chat de cualquier proveedor, plantillas de prompts, extracción de salida estructurada, herramientas, recuperadores de documentos y una forma de **componer** todo con el operador `|`. Desde su versión 1.0 incluye también `create_agent`, que arma un agente en pocas líneas.
- **LangGraph** aporta el **motor de ejecución**: modela una aplicación como un **grafo con estado**, con nodos (funciones), aristas (transiciones, incluidas las condicionales y los ciclos), memoria persistente, pausas para la intervención humana y *streaming*. Los agentes necesitan ciclos (*pensar → actuar → observar → repetir*), y LangGraph es el motor sobre el que corre `create_agent`.

```text
                          Tu aplicación
  ┌──────────────────────────────────────────────────────────────┐
  │  langchain            create_agent y middleware              │
  │  langchain-core       Runnables (LCEL), mensajes, prompts,   │  componentes
  │                       herramientas, parsers, vector stores   │
  │  langgraph            grafos con estado, memoria,            │  motor de ejecución
  │                       interrupciones, streaming              │
  │  langchain-openai · langchain-google-genai · langchain-ollama│  un paquete por proveedor
  └──────────────────────────────────────────────────────────────┘
        LangSmith: servicio opcional de trazas y evaluación
```

En este cuaderno:

1. Usamos los mismos tres proveedores del cuaderno 18 (OpenAI, Google y Ollama) a través de **una sola interfaz**.
2. Construimos cadenas con **prompts, salida estructurada y `|`** (LCEL).
3. Estudiamos el ***tool calling*** en detalle: cómo un modelo "usa" herramientas, el bucle completo escrito a mano, los errores y cómo medir si el modelo elige bien la herramienta.
4. Montamos un **RAG** y aprendemos los fundamentos de **LangGraph**: estado, nodos, aristas, ciclos, memoria e intervención humana.
5. Creamos **agentes**: primero desde cero con LangGraph, luego con `create_agent`; los controlamos, los hacemos colaborar y medimos su costo.

Una advertencia sobre la **volatilidad**: estas librerías evolucionan muy rápido (la creación de agentes pasó de `langgraph.prebuilt.create_react_agent` a `langchain.agents.create_agent`, y hay métodos que ya se marcan como obsoletos). Por eso el cuaderno imprime las versiones usadas; si algo falla con una versión distinta, revisa la documentación oficial.

<a id="configuracion-y-proveedores"></a>
<h2 style="color: #007ACC;">Configuración y proveedores</h2>

Usamos las **mismas claves** del cuaderno 18, guardadas en el archivo `.env` de la raíz del proyecto: `OPENAI_API_KEY` y `GOOGLE_API_KEY` (en Colab, en el panel *Secretos* con los mismos nombres). Ollama no necesita clave: solo el servidor local con un modelo descargado (`ollama pull llama3.2:3b`). Las claves **nunca se imprimen** ni se escriben en el cuaderno.

Cada proveedor tiene su paquete de integración con una clase de modelo de chat:

| Proveedor | Paquete | Clase |
|---|---|---|
| OpenAI | `langchain-openai` | `ChatOpenAI` |
| Google (Gemini) | `langchain-google-genai` | `ChatGoogleGenerativeAI` |
| Ollama | `langchain-ollama` | `ChatOllama` |

Las tres comparten la misma interfaz (`BaseChatModel`), así que el resto del cuaderno usa el mismo código para cualquiera. También existe `init_chat_model("proveedor:modelo")`, que elige la clase por el nombre; lo vemos abajo. El cuaderno se ejecuta con los proveedores que tengas disponibles.

In [3]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # busca el .env en la carpeta actual y en las superiores


def obtener_clave(nombre):
    """Lee la clave del entorno (.env) o, en Google Colab, de los Secretos. Nunca se imprime."""
    clave = os.environ.get(nombre)
    if not clave:
        try:
            from google.colab import userdata
            clave = userdata.get(nombre)
        except Exception:
            clave = None
    return clave


# Los nombres de los modelos cambian con frecuencia: si alguno falla, consulta la lista del proveedor y cámbialo aquí.
MODELO_OPENAI = "gpt-4.1-mini"
MODELO_GOOGLE = "gemini-3.5-flash-lite"
MODELO_OLLAMA = "llama3.2:3b"

CLAVE_OPENAI = obtener_clave("OPENAI_API_KEY")
CLAVE_GOOGLE = obtener_clave("GOOGLE_API_KEY")
# Las integraciones leen las claves del entorno; en Colab hay que copiarlas allí.
if CLAVE_OPENAI:
    os.environ["OPENAI_API_KEY"] = CLAVE_OPENAI
if CLAVE_GOOGLE:
    os.environ["GOOGLE_API_KEY"] = CLAVE_GOOGLE


def ollama_listo():
    """¿Está el servidor de Ollama activo y el modelo descargado?"""
    try:
        import ollama
        return any(m.model == MODELO_OLLAMA for m in ollama.Client().list().models)
    except Exception:
        return False


estado = {
    "OpenAI": (MODELO_OPENAI, "lista" if CLAVE_OPENAI else "falta OPENAI_API_KEY"),
    "Google": (MODELO_GOOGLE, "lista" if CLAVE_GOOGLE else "falta GOOGLE_API_KEY"),
    "Ollama": (MODELO_OLLAMA, "lista" if ollama_listo() else "servidor apagado o modelo sin descargar"),
}
display(pd.DataFrame(estado, index=["modelo", "estado"]).T)
DISPONIBLES = [proveedor for proveedor, (_, e) in estado.items() if e == "lista"]
assert DISPONIBLES, "Configura al menos un proveedor (ver arriba) para ejecutar el resto del cuaderno."
print("Proveedores que usaremos:", DISPONIBLES)

,modelo,estado
OpenAI,gpt-4.1-mini,lista
Google,gemini-3.5-flash-lite,lista
Ollama,llama3.2:3b,lista


Proveedores que usaremos: ['OpenAI', 'Google', 'Ollama']


In [4]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from langchain.chat_models import init_chat_model


def crear_modelo(proveedor, temperatura=0.0):
    """Un modelo de chat de LangChain para el proveedor indicado."""
    if proveedor == "OpenAI":
        return ChatOpenAI(model=MODELO_OPENAI, temperature=temperatura)
    if proveedor == "Google":
        return ChatGoogleGenerativeAI(model=MODELO_GOOGLE)  # sin temperatura: este modelo la ignora (ver abajo)
    if proveedor == "Ollama":
        return ChatOllama(model=MODELO_OLLAMA, temperature=temperatura)
    raise ValueError(proveedor)


MODELOS = {proveedor: crear_modelo(proveedor) for proveedor in DISPONIBLES}
PRINCIPAL = DISPONIBLES[0]  # el proveedor con el que corren las demostraciones de un solo modelo
modelo = MODELOS[PRINCIPAL]
print("Modelo principal:", PRINCIPAL, "->", type(modelo).__name__)

# Equivalente con init_chat_model: el prefijo del proveedor elige la clase.
equivalentes = {"OpenAI": f"openai:{MODELO_OPENAI}", "Google": f"google_genai:{MODELO_GOOGLE}", "Ollama": f"ollama:{MODELO_OLLAMA}"}
print({p: type(init_chat_model(equivalentes[p])).__name__ for p in DISPONIBLES})

Modelo principal: OpenAI -> ChatOpenAI
{'OpenAI': 'ChatOpenAI', 'Google': 'ChatGoogleGenerativeAI', 'Ollama': 'ChatOllama'}


**Una nota sobre la temperatura de Gemini.** Si pasamos `temperature` a `ChatGoogleGenerativeAI` con este modelo, LangChain avisa de que `gemini-3.5-flash-lite` *"usa valores de muestreo fijos y el parámetro de temperatura se ignorará"*. Es la razón por la que en el cuaderno 18 el experimento de temperatura de Google devolvió cinco eslóganes distintos incluso con $T = 0$: **el modelo no estaba respetando el parámetro**. Por eso `crear_modelo` no se lo pasa. Es un ejemplo de algo que una librería puede avisar y un SDK usado a mano no: no todos los modelos aceptan todos los parámetros.

<a id="modelos-de-chat"></a>
<h2 style="color: #007ACC;">Modelos de chat</h2>

Un **modelo de chat** de LangChain recibe una lista de **mensajes** y devuelve un mensaje. Los tipos de mensaje (en `langchain_core.messages`) son los roles del cuaderno 18, convertidos en clases:

| Clase | Rol | Contenido |
|---|---|---|
| `SystemMessage` | `system` | Instrucciones generales para el modelo |
| `HumanMessage` | `user` | Lo que dice el usuario |
| `AIMessage` | `assistant` | La respuesta del modelo: texto y, si pide usar herramientas, `tool_calls` |
| `ToolMessage` | `tool` | El resultado de una herramienta, ligado a la petición por `tool_call_id` |

El método principal es `invoke`, que acepta un texto, una lista de mensajes o una lista de diccionarios `{"role": ..., "content": ...}`. El resultado es un `AIMessage` con el texto en `.text`, el consumo de tokens en `.usage_metadata` y los datos propios del proveedor en `.response_metadata`. Compara con el cuaderno 18: lo que allí escribimos a mano (el objeto `Respuesta` con tokens, la función por proveedor) aquí viene incluido.

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

mensajes = [SystemMessage("Responde en una sola frase."), HumanMessage("¿Qué es un grafo de estados?")]
filas = []
for proveedor, m in MODELOS.items():
    r = m.invoke(mensajes)
    print(f"[{proveedor}] {r.text}\n")
    filas.append({"proveedor": proveedor, "clase": type(r).__name__, "tipo de .content": type(r.content).__name__,
                  "tokens (entrada/salida)": f"{r.usage_metadata['input_tokens']}/{r.usage_metadata['output_tokens']}",
                  "modelo reportado": r.response_metadata.get("model_name") or r.response_metadata.get("model")})
pd.DataFrame(filas).set_index("proveedor")

[OpenAI] Un grafo de estados es una representación gráfica que muestra los diferentes estados de un sistema y las transiciones entre ellos.



[Google] Un grafo de estados es una representación matemática compuesta por nodos que simbolizan las diferentes situaciones posibles de un sistema y aristas que indican las transiciones válidas entre ellas.



[Ollama] Un grafo de estados es una estructura matemática que representa una secuencia de estados en el tiempo, donde cada estado se representa como un vértice y las transiciones entre ellos se representan como aristas.



,clase,tipo de .content,tokens (entrada/salida),modelo reportado
proveedor,,,,
OpenAI,AIMessage,str,26/24,gpt-4.1-mini-2025-04-14
Google,AIMessage,list,16/34,gemini-3.5-flash-lite
Ollama,AIMessage,str,41/47,llama3.2:3b


Fíjate en la columna `tipo de .content`: según el proveedor, `.content` puede ser un **texto** o una **lista de bloques** (por ejemplo, `[{"type": "text", "text": "..."}]`, que es lo que devuelve Gemini). Por eso usamos siempre **`.text`**, que extrae el texto en cualquier caso. Es un pequeño ejemplo de por qué conviene usar la interfaz común y no el contenido crudo.

Los modelos de chat también son *Runnables* (los estudiamos con LCEL más abajo), así que además de `invoke` tienen `stream` (recibir la respuesta por trozos), `batch` (varias entradas en paralelo) y las versiones asíncronas `ainvoke`, `astream` y `abatch`.

In [6]:
# stream: la respuesta llega por trozos (útil para mostrar texto mientras se genera)
trozos = [t.text for t in modelo.stream("Cuenta del 1 al 5, separados por comas.")]
print(f"{len(trozos)} trozos -> {''.join(trozos)!r}")

# batch: varias entradas a la vez, con un máximo de llamadas simultáneas
preguntas = ["Capital de Colombia", "Capital de Perú", "Capital de Chile", "Capital de Ecuador"]
t = time.perf_counter()
respuestas = modelo.batch([f"{p}. Responde solo con el nombre." for p in preguntas], config={"max_concurrency": 4})
print([r.text.strip() for r in respuestas], f"({time.perf_counter() - t:.1f} s)")

17 trozos -> '1, 2, 3, 4, 5'


['Bogotá', 'Lima', 'Santiago', 'Quito'] (1.4 s)


<a id="plantillas-de-prompt"></a>
<h2 style="color: #007ACC;">Plantillas de prompt</h2>

Un prompt real casi nunca es un texto fijo: tiene **partes variables** (la pregunta del usuario, los documentos recuperados, el historial). Una **plantilla de prompt** separa la estructura de los datos:

- `ChatPromptTemplate.from_messages([...])` define una lista de mensajes con variables entre llaves, como `{idioma}`.
- `.invoke({...})` rellena las variables y devuelve un `PromptValue`, que se convierte en mensajes con `.to_messages()`.
- `.partial(...)` fija algunas variables de antemano.
- `MessagesPlaceholder("nombre")` reserva un lugar para una **lista de mensajes** (por ejemplo, el historial de la conversación o ejemplos *few-shot*).

La plantilla es un *Runnable* más: recibe un diccionario y devuelve el prompt, listo para conectarse a un modelo.

In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

plantilla = ChatPromptTemplate.from_messages([
    ("system", "Eres un traductor profesional al {idioma}. Responde solo con la traducción."),
    MessagesPlaceholder("ejemplos", optional=True),   # lugar para ejemplos few-shot (opcional)
    ("human", "{texto}"),
])

valor = plantilla.invoke({"idioma": "inglés", "texto": "Buenos días"})
for m in valor.to_messages():
    print(f"{type(m).__name__:<14}{m.content}")

# ejemplos few-shot: pares humano / IA que el modelo imita
ejemplos = [HumanMessage("Hasta luego"), AIMessage("See you later")]
print("\ncon ejemplos:", [type(m).__name__ for m in plantilla.invoke({"idioma": "inglés", "texto": "Gracias", "ejemplos": ejemplos}).to_messages()])

# partial: fija una variable de antemano
a_frances = plantilla.partial(idioma="francés")
print("\nvariables que faltan en la plantilla parcial:", a_frances.input_variables)

SystemMessage Eres un traductor profesional al inglés. Responde solo con la traducción.
HumanMessage  Buenos días

con ejemplos: ['SystemMessage', 'HumanMessage', 'AIMessage', 'HumanMessage']

variables que faltan en la plantilla parcial: ['texto']


<a id="salida-estructurada"></a>
<h2 style="color: #007ACC;">Salida estructurada</h2>

En el cuaderno 18 vimos que cada proveedor impone un esquema de forma distinta (`response_format`, `response_schema`, `format`). LangChain unifica los tres con un único método: **`modelo.with_structured_output(Esquema)`**. Devuelve un nuevo *Runnable* que, al invocarlo, entrega directamente una instancia de la clase de **Pydantic** (o un diccionario, si el esquema es un `TypedDict` o un JSON Schema), ya validada. Por dentro usa el mecanismo nativo de cada proveedor (herramientas, o JSON con esquema).

Con `include_raw=True` devuelve además el mensaje original del modelo y los posibles errores de análisis. Recuerda lo que vimos en el cuaderno 18: el esquema garantiza la **forma**, no el **contenido**.

In [8]:
class Queja(BaseModel):
    producto: str = Field(description="Producto o servicio del que se queja el cliente")
    problema: Literal["defecto", "envio", "servicio", "precio", "otro"] = Field(description="Tipo principal de problema")
    urgencia: int = Field(description="De 1 (baja) a 5 (muy urgente)")
    pide_reembolso: bool = Field(description="Verdadero si el cliente exige que le devuelvan el dinero")


QUEJAS = [
    "Compré una licuadora hace dos semanas y ya no enciende. Llevo cinco días esperando respuesta del vendedor. Quiero que me devuelvan el dinero, es urgente.",
    "El paquete llegó con tres días de retraso, pero el producto está perfecto. Solo quería avisar, gracias.",
]
for queja in QUEJAS:
    print(f"\n«{queja[:90]}...»")
    filas = {p: m.with_structured_output(Queja).invoke(queja).model_dump() for p, m in MODELOS.items()}  # una línea por proveedor
    display(pd.DataFrame(filas).T)

# include_raw: el mensaje original, el objeto analizado y el posible error
crudo = modelo.with_structured_output(Queja, include_raw=True).invoke(QUEJAS[0])
print("claves:", list(crudo), "| error de análisis:", crudo["parsing_error"], "| tipo del mensaje original:", type(crudo["raw"]).__name__)


«Compré una licuadora hace dos semanas y ya no enciende. Llevo cinco días esperando respues...»


,producto,problema,urgencia,pide_reembolso
OpenAI,licuadora,defecto,5,True
Google,licuadora,defecto,5,True
Ollama,licuadora,defecto,5,True



«El paquete llegó con tres días de retraso, pero el producto está perfecto. Solo quería avi...»


,producto,problema,urgencia,pide_reembolso
OpenAI,paquete,envio,2,False
Google,paquete,envio,1,False
Ollama,perfecto,defecto,0,False


claves: ['raw', 'parsed', 'parsing_error'] | error de análisis: None | tipo del mensaje original: AIMessage


Los tres proveedores devuelven un objeto válido con **una sola línea** por proveedor (compara con las tres funciones distintas del cuaderno 18). Pero el esquema garantiza la **forma**, no el **contenido**: en la segunda queja, el correo que agradece un producto que *"está perfecto"*, el modelo local pequeño produce un objeto válido y aun así se equivoca (`producto='perfecto'`, `problema='defecto'` y `urgencia=0`, fuera del rango de 1 a 5 que solo estaba descrito en el texto del campo).

<a id="lcel-composicion-con-runnables"></a>
<h2 style="color: #007ACC;">LCEL: composición con Runnables</h2>

Todo lo anterior (plantillas, modelos, extractores de salida) es un ***Runnable***: un componente con una interfaz común. Esa uniformidad permite **componerlos con el operador `|`**, en lo que LangChain llama *LangChain Expression Language* (LCEL):

```python
cadena = plantilla | modelo | StrOutputParser()
```

La salida de cada componente es la entrada del siguiente. Y la cadena resultante es otro *Runnable*, con **la misma interfaz**:

| Método | Qué hace |
|---|---|
| `invoke(x)` / `ainvoke(x)` | Una entrada, una salida (síncrono / asíncrono) |
| `stream(x)` / `astream(x)` | La salida por trozos, a medida que se genera |
| `batch([x, y])` / `abatch` | Varias entradas en paralelo (`max_concurrency` limita la simultaneidad) |
| `with_retry()` | Reintenta ante errores transitorios |
| `with_fallbacks([otro])` | Si falla, prueba con otro componente (por ejemplo, otro modelo) |
| `with_config(...)` | Fija nombre, etiquetas o callbacks para las trazas |

Además de los modelos y las plantillas, hay *Runnables* para armar la lógica de la cadena:

- `RunnableLambda(f)`: convierte cualquier función de Python en un componente.
- `RunnablePassthrough`: deja pasar la entrada tal cual; `.assign(clave=...)` le añade campos calculados.
- `RunnableParallel({...})` (o un simple diccionario dentro de la cadena): ejecuta varias ramas **en paralelo** con la misma entrada y devuelve un diccionario.

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel

traducir = plantilla | modelo | StrOutputParser()          # prompt | modelo | parser
print("invoke :", traducir.invoke({"idioma": "inglés", "texto": "Buenos días, ¿cómo estás?"}))
print("batch  :", traducir.batch([{"idioma": "francés", "texto": "gracias"}, {"idioma": "alemán", "texto": "gracias"}]))
print("stream :", "".join(traducir.stream({"idioma": "italiano", "texto": "buenas noches"})))

print("\nLa cadena como grafo:")
traducir.get_graph().print_ascii()

invoke : Good morning, how are you?


batch  : ['merci', 'Danke']


stream : buona notte

La cadena como grafo:
     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


In [10]:
# Ramas en paralelo + campos calculados con assign
a_ingles = plantilla.partial(idioma="inglés") | modelo | StrOutputParser()
a_frances = plantilla.partial(idioma="francés") | modelo | StrOutputParser()

dos_idiomas = RunnableParallel(ingles=a_ingles, frances=a_frances)
cadena = dos_idiomas | RunnablePassthrough.assign(
    longitudes=RunnableLambda(lambda d: {k: len(v) for k, v in d.items()}))   # usa el resultado de las ramas

print(cadena.invoke({"texto": "El conocimiento es poder."}))

{'ingles': 'Knowledge is power.', 'frances': 'Le savoir est le pouvoir.', 'longitudes': {'ingles': 19, 'frances': 25}}


In [11]:
# Clasificador de sentimiento de reseñas: lo que en el cuaderno 18 escribimos a mano, en una cadena corta
class Sentimiento(BaseModel):
    etiqueta: Literal["positiva", "negativa", "neutra"]


clasificador = (
    ChatPromptTemplate.from_messages([
        ("system", "Clasificas el sentimiento de reseñas de productos en español. La reseña está entre <reseña> y </reseña>."),
        ("human", "<reseña>\n{texto}\n</reseña>"),
    ])
    | modelo.with_structured_output(Sentimiento)
)
RESENAS = ["Excelente producto, llegó rápido y funciona perfecto.", "Se rompió a la semana, una pérdida de dinero.",
           "Cumple, nada especial.", "Me encantó, lo recomiendo a todos.", "No funciona y el vendedor no responde.",
           "Está bien por el precio, aunque esperaba algo más de calidad."]
etiquetas = [r.etiqueta for r in clasificador.batch([{"texto": t} for t in RESENAS], config={"max_concurrency": 4})]
pd.DataFrame({"reseña": RESENAS, "etiqueta": etiquetas})

,reseña,etiqueta
0,"Excelente producto, llegó rápido y funciona perfecto.",positiva
1,"Se rompió a la semana, una pérdida de dinero.",negativa
2,"Cumple, nada especial.",neutra
3,"Me encantó, lo recomiendo a todos.",positiva
4,No funciona y el vendedor no responde.,negativa
5,"Está bien por el precio, aunque esperaba algo más de calidad.",neutra


**Tolerancia a fallos.** Las APIs fallan: límites de tasa, cortes de red, modelos retirados. Dos modificadores lo resuelven sin escribir bucles:

- `with_retry(stop_after_attempt=n)` reintenta el mismo componente.
- `with_fallbacks([alternativo, ...])` pasa a otro componente cuando el primero lanza una excepción. Es la forma natural de usar **varios proveedores como respaldo**: un modelo principal en la nube y uno local de reserva.

**Concurrencia.** Como las llamadas a un LLM son de espera, conviene lanzarlas en paralelo. Comparamos seis llamadas de forma secuencial, con `batch` y con `ainvoke` dentro de `asyncio.gather`:

In [12]:
# Un modelo con nombre inexistente falla; with_fallbacks pasa al modelo bueno
roto = ChatOpenAI(model="modelo-que-no-existe", max_retries=0) if "OpenAI" in MODELOS else ChatOllama(model="modelo-que-no-existe")
con_respaldo = roto.with_fallbacks([modelo]) | StrOutputParser()
print("con respaldo:", con_respaldo.invoke("Responde solo con la palabra: listo"))

# secuencial vs batch vs asíncrono
entradas = [f"Di el número {i}. Responde solo con el número." for i in range(6)]
t = time.perf_counter(); [modelo.invoke(e) for e in entradas]; t_sec = time.perf_counter() - t
t = time.perf_counter(); modelo.batch(entradas, config={"max_concurrency": 6}); t_batch = time.perf_counter() - t
t = time.perf_counter(); await asyncio.gather(*[modelo.ainvoke(e) for e in entradas]); t_async = time.perf_counter() - t
pd.Series({"secuencial": t_sec, "batch": t_batch, "ainvoke + gather": t_async}, name="segundos (6 llamadas)").round(2).to_frame()

con respaldo: listo


,segundos (6 llamadas)
secuencial,3.60
batch,0.78
ainvoke + gather,0.80


Seis llamadas tardaron unos 3,6 s una tras otra y menos de 1 s con `batch` o con `ainvoke` en paralelo, más de cuatro veces menos: el tiempo de una llamada es de **espera de red**, así que se solapa bien. Con APIs reales la concurrencia choca con los **límites de tasa** del proveedor, y `max_concurrency` permite acotarla.

<a id="tool-calling"></a>
<h2 style="color: #007ACC;">Tool calling</h2>

Un LLM, por sí solo, solo **produce texto**: no puede consultar el clima, buscar en una base de datos ni calcular con fiabilidad. El ***tool calling*** (llamada a herramientas, o *function calling*) le permite pedirle a tu programa que ejecute acciones. Es la base de los agentes.

**El modelo no ejecuta nada.** Lo que hace es **decidir** qué herramienta necesita y con qué argumentos, y emitir esa decisión como datos estructurados. Quien ejecuta es **tu código**. El flujo completo es:

```text
 1. Tú → modelo:   la pregunta + las descripciones de las herramientas disponibles (nombre, descripción, esquema de argumentos)
 2. Modelo → tú:   AIMessage con tool_calls = [{"name": "clima", "args": {"ciudad": "Tunja"}, "id": "call_1"}]
 3. Tu código:     ejecuta clima("Tunja") y obtiene "14 °C, llovizna"
 4. Tú → modelo:   la conversación + un ToolMessage(content="14 °C, llovizna", tool_call_id="call_1")
 5. Modelo → tú:   la respuesta final en lenguaje natural   (o nuevos tool_calls: se repite desde el paso 3)
```

Tres ideas clave:

- El modelo **solo ve descripciones**: el nombre, el docstring y el esquema de argumentos de cada herramienta. De ellos depende que elija bien (lo medimos más abajo).
- El `id` de cada `tool_call` es el que enlaza la petición con su resultado, y permite que el modelo pida **varias herramientas a la vez** (llamadas paralelas).
- El paso 5 puede pedir más herramientas. Ese **bucle** (modelo → herramientas → modelo → ...) es lo que convierte un modelo en un agente.

Empezamos definiendo las herramientas. Usamos datos de ejemplo fijos (clima, tasas de cambio y un pequeño corpus de documentos) para que los resultados sean deterministas y verificables.

In [13]:
import ast
import operator as op
import unicodedata

from langchain_core.tools import tool, StructuredTool

TASAS = {"USD": 4100.0, "EUR": 4500.0, "COP": 1.0}  # pesos por unidad de cada moneda (tasas fijas de ejemplo)
CLIMAS = {"tunja": "14 °C, llovizna", "bogotá": "16 °C, nublado", "cali": "28 °C, soleado", "duitama": "15 °C, nublado"}


class Conversion(BaseModel):
    """Argumentos de convertir_moneda."""
    monto: float = Field(description="Cantidad a convertir")
    origen: Literal["USD", "EUR", "COP"] = Field(description="Moneda de origen")
    destino: Literal["USD", "EUR", "COP"] = Field(description="Moneda de destino")


@tool
def clima(ciudad: str) -> str:
    """Devuelve el clima actual (temperatura y condición) de una ciudad colombiana."""
    return f"En {ciudad}: {CLIMAS.get(ciudad.lower(), 'sin datos')}."


@tool(args_schema=Conversion)
def convertir_moneda(monto: float, origen: str, destino: str) -> float:
    """Convierte un monto de una moneda a otra (USD, EUR o COP) con las tasas de cambio del día."""
    return round(monto * TASAS[origen] / TASAS[destino], 2)


OPERADORES = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv, ast.Pow: op.pow, ast.USub: op.neg}


def _evaluar(nodo):
    if isinstance(nodo, ast.Constant) and isinstance(nodo.value, (int, float)):
        return nodo.value
    if isinstance(nodo, ast.BinOp) and type(nodo.op) in OPERADORES:
        return OPERADORES[type(nodo.op)](_evaluar(nodo.left), _evaluar(nodo.right))
    if isinstance(nodo, ast.UnaryOp) and type(nodo.op) in OPERADORES:
        return OPERADORES[type(nodo.op)](_evaluar(nodo.operand))
    raise ValueError("Expresión no permitida: solo números y + - * / **")


@tool
def calculadora(expresion: str) -> str:
    """Evalúa una expresión aritmética, por ejemplo '12*(3+4)'. Úsala para cualquier cálculo numérico."""
    return str(_evaluar(ast.parse(expresion, mode="eval").body))  # nunca eval(): solo se aceptan operaciones aritméticas


PARRAFOS = [
    "El curso de Procesamiento de Lenguaje Natural pertenece a la Especialización en Inteligencia Artificial de la UPTC y se organiza en cuatro módulos de dos semanas cada uno.",
    "Módulo 1 (semanas 1 y 2): introducción al PLN, preprocesamiento de texto y tokenización, desambiguación y etiquetado, análisis de sentimiento con regresión logística y Naive Bayes, y espacios vectoriales.",
    "Módulo 2 (semanas 3 y 4): métodos probabilísticos: Bag-of-Words y N-gramas, clasificación con NLTK, modelos ocultos de Markov, algoritmo de Viterbi y modelos de Markov de máxima entropía (MEMM).",
    "Módulo 3 (semanas 5 y 6): modelos secuenciales: embeddings con redes neuronales, Word2Vec y GloVe, RNN, LSTM y GRU, y modelos Seq2Seq.",
    "Módulo 4 (semanas 7 y 8): atención y modelos preentrenados: Transformers, BERT y GPT con fine-tuning, aplicaciones de modelos preentrenados, prompt engineering, LangChain y LangGraph, y ética en PLN.",
    "Cada módulo tiene cuadernos de clase, talleres prácticos con su solución y un proyecto de aplicación.",
    "El proyecto del módulo 4 se desarrolla en grupos de 2 a 3 personas y debe poder ejecutarse en Google Colab con una GPU T4.",
    "El proyecto del módulo 4 exige una presentación de 10 minutos con los hallazgos principales y una auditoría ética con su tarjeta del modelo.",
    "Los requisitos de acceso son Aprendizaje Automático, Ciencia de Datos y Aprendizaje Automático para la Toma de Decisiones, y Aprendizaje Profundo; se espera experiencia avanzada en Python.",
    "Se recomienda seguir el orden numérico de los cuadernos y resolver el taller de cada módulo antes de pasar al siguiente.",
]


def _palabras(texto):
    sin_tildes = "".join(c for c in unicodedata.normalize("NFD", texto.lower()) if unicodedata.category(c) != "Mn")
    return {p for p in re.findall(r"[a-z0-9]+", sin_tildes) if len(p) > 3}


@tool
def buscar_documentos(consulta: str) -> str:
    """Busca información en los documentos del curso de PLN: programa, módulos, semanas, proyectos y requisitos."""
    puntajes = sorted(((len(_palabras(consulta) & _palabras(p)), p) for p in PARRAFOS), reverse=True)  # coincidencia de palabras
    return "\n".join(p for puntaje, p in puntajes[:2] if puntaje > 0) or "Sin resultados."


HERRAMIENTAS_BASE = [clima, convertir_moneda, calculadora, buscar_documentos]
print("Herramientas:", [h.name for h in HERRAMIENTAS_BASE])

Herramientas: ['clima', 'convertir_moneda', 'calculadora', 'buscar_documentos']


**Anatomía de una herramienta.** El decorador `@tool` convierte una función en un objeto con tres cosas que el modelo necesita:

- **`name`**: el nombre de la función.
- **`description`**: el **docstring**. Es lo más importante: es lo único que el modelo lee para decidir si usarla.
- **`args_schema`**: el esquema de los argumentos, generado a partir de las **anotaciones de tipo**, o dado explícitamente con una clase de Pydantic (como `Conversion`), que permite añadir descripciones por campo y restricciones (`Literal` produce una lista de valores permitidos).

Veamos exactamente lo que recibe el proveedor, y cómo se invoca una herramienta directamente, sin modelo.

In [14]:
from langchain_core.utils.function_calling import convert_to_openai_tool

print("name:", convertir_moneda.name, "\ndescription:", convertir_moneda.description)
print("\nEsquema que recibe el modelo (formato de herramienta de OpenAI):")
print(json.dumps(convert_to_openai_tool(convertir_moneda), indent=2, ensure_ascii=False))

print("\nInvocación directa (sin modelo):")
print("  calculadora :", calculadora.invoke({"expresion": "12*(3+4)"}))
print("  convertir   :", convertir_moneda.invoke({"monto": 250, "origen": "USD", "destino": "COP"}))
for argumentos in [{"expresion": "__import__('os').getcwd()"}]:
    try:
        calculadora.invoke(argumentos)
    except Exception as error:
        print(f"  calculadora con código malicioso -> {type(error).__name__}: {error}")
try:
    convertir_moneda.invoke({"monto": 10, "origen": "GBP", "destino": "COP"})
except Exception as error:
    print(f"  convertir con una moneda no permitida -> {type(error).__name__}: {str(error).splitlines()[2].strip()[:70]}")

name: convertir_moneda 
description: Convierte un monto de una moneda a otra (USD, EUR o COP) con las tasas de cambio del día.

Esquema que recibe el modelo (formato de herramienta de OpenAI):
{
  "type": "function",
  "function": {
    "name": "convertir_moneda",
    "description": "Convierte un monto de una moneda a otra (USD, EUR o COP) con las tasas de cambio del día.",
    "parameters": {
      "properties": {
        "monto": {
          "description": "Cantidad a convertir",
          "type": "number"
        },
        "origen": {
          "description": "Moneda de origen",
          "enum": [
            "USD",
            "EUR",
            "COP"
          ],
          "type": "string"
        },
        "destino": {
          "description": "Moneda de destino",
          "enum": [
            "USD",
            "EUR",
            "COP"
          ],
          "type": "string"
        }
      },
      "required": [
        "monto",
        "origen",
        "destino"
      ],

Dos observaciones sobre la celda anterior. Primero, la **calculadora no usa `eval`**: evalúa solo operaciones aritméticas sobre el árbol sintáctico, así que un argumento malicioso (como `__import__('os')...`) se rechaza. Con una herramienta que ejecuta lo que el modelo escribe, el modelo (o quien lo manipule por inyección de prompts) controla los argumentos. Segundo, el esquema valida los argumentos **antes** de ejecutar: una moneda fuera de la lista lanza un `ValidationError` sin llegar a la función.

**`bind_tools`.** Para que un modelo conozca las herramientas, se le *enlazan* con `bind_tools([...])`, que devuelve un modelo nuevo. Al invocarlo, si el modelo decide usar herramientas, el `AIMessage` trae la lista `tool_calls` (con `name`, `args` e `id`) en lugar de texto. Hacemos una pregunta que requiere **dos herramientas independientes** a los tres proveedores:

In [15]:
pregunta = "Dime el clima de Tunja y cuántos pesos son 250 USD."
filas = []
for proveedor, m in MODELOS.items():
    r = m.bind_tools(HERRAMIENTAS_BASE).invoke(pregunta)
    for llamada in r.tool_calls:
        filas.append({"proveedor": proveedor, "herramienta": llamada["name"], "argumentos": llamada["args"], "id": llamada["id"][:12]})
    print(f"[{proveedor}] texto en la respuesta: {r.text!r} | llamadas: {len(r.tool_calls)}")
display(pd.DataFrame(filas).set_index(["proveedor", "herramienta"]))

[OpenAI] texto en la respuesta: '' | llamadas: 2


[Google] texto en la respuesta: '' | llamadas: 2


[Ollama] texto en la respuesta: '' | llamadas: 2


argumentos  \
proveedor herramienta                                                             
OpenAI    clima                                             {'ciudad': 'Tunja'}   
          convertir_moneda    {'monto': 250, 'origen': 'USD', 'destino': 'COP'}   
Google    clima                                             {'ciudad': 'Tunja'}   
          convertir_moneda    {'monto': 250, 'origen': 'USD', 'destino': 'COP'}   
Ollama    clima                                             {'ciudad': 'Tunja'}   
          convertir_moneda  {'origen': 'USD', 'destino': 'COP', 'monto': '250'}   

                                      id  
proveedor herramienta                     
OpenAI    clima             call_LBd4ziy  
          convertir_moneda  call_PlCkHqE  
Google    clima              call_346131  
          convertir_moneda   call_346132  
Ollama    clima             b089344a-b0d  
          convertir_moneda  21f78721-409

Los tres proveedores devuelven la **misma estructura**, y el modelo pide las dos herramientas **a la vez** (llamadas paralelas) con el campo `content` vacío: aún no hay texto que mostrar, solo una petición. Algunas diferencias de detalle: Ollama entrega los números como texto (`'250'`) y los otros como números; el esquema de la herramienta se encarga de convertirlos al ejecutar.

**El bucle completo, escrito a mano.** Para ejecutar las herramientas y devolverle los resultados al modelo, hacemos el bucle de los pasos 3 a 5. La función de abajo es, en esencia, lo que hace un agente: repite *modelo → herramientas → modelo* hasta que el modelo responde sin pedir más herramientas, con un **máximo de pasos** para que no entre en un ciclo infinito.

In [16]:
def ejecutar_con_herramientas(modelo, herramientas, pregunta, sistema=None, max_pasos=6):
    """Bucle modelo -> herramientas -> modelo, escrito a mano. Devuelve la lista completa de mensajes."""
    por_nombre = {h.name: h for h in herramientas}
    modelo_h = modelo.bind_tools(herramientas)
    mensajes = ([SystemMessage(sistema)] if sistema else []) + [HumanMessage(pregunta)]
    for _ in range(max_pasos):
        respuesta = modelo_h.invoke(mensajes)
        mensajes.append(respuesta)
        if not respuesta.tool_calls:                 # el modelo responde con texto: terminamos
            return mensajes
        for llamada in respuesta.tool_calls:         # una o varias herramientas en paralelo
            try:
                resultado = por_nombre[llamada["name"]].invoke(llamada)  # invocar con la tool_call devuelve un ToolMessage
            except Exception as error:               # un error de la herramienta se le cuenta al modelo
                resultado = ToolMessage(content=f"Error: {type(error).__name__}: {error}", tool_call_id=llamada["id"],
                                        name=llamada["name"], status="error")
            mensajes.append(resultado)
    raise RuntimeError(f"Se superó el máximo de {max_pasos} pasos")


def mostrar(mensajes):
    """Imprime la conversación de forma compacta."""
    for m in mensajes:
        if isinstance(m, SystemMessage):
            continue
        if isinstance(m, HumanMessage):
            print(f"usuario      | {m.text}")
        elif isinstance(m, AIMessage) and m.tool_calls:
            print("modelo       | pide: " + "; ".join(f"{c['name']}({c['args']})" for c in m.tool_calls))
        elif isinstance(m, AIMessage):
            print(f"modelo       | {m.text.strip()}")
        elif isinstance(m, ToolMessage):
            print(f"herramienta  | {m.name} -> {m.content}" + ("  [ERROR]" if m.status == "error" else ""))


mensajes = ejecutar_con_herramientas(modelo, HERRAMIENTAS_BASE, pregunta)
print(f"Conversación completa ({PRINCIPAL}):")
mostrar(mensajes)
print("\nRespuesta final de cada proveedor:")
for proveedor, m in MODELOS.items():
    print(f"  [{proveedor}] {ejecutar_con_herramientas(m, HERRAMIENTAS_BASE, pregunta)[-1].text.strip()[:200]}")

Conversación completa (OpenAI):
usuario      | Dime el clima de Tunja y cuántos pesos son 250 USD.
modelo       | pide: clima({'ciudad': 'Tunja'}); convertir_moneda({'monto': 250, 'origen': 'USD', 'destino': 'COP'})
herramienta  | clima -> En Tunja: 14 °C, llovizna.
herramienta  | convertir_moneda -> 1025000.0
modelo       | El clima en Tunja es de 14 °C con llovizna. Además, 250 USD equivalen a 1,025,000 pesos colombianos. ¿Quieres saber algo más?

Respuesta final de cada proveedor:


  [OpenAI] El clima en Tunja es de 14 °C con llovizna. Además, 250 USD equivalen a 1,025,000 pesos colombianos. ¿Quieres saber algo más?


  [Google] El clima actual en Tunja es de **14 °C con llovizna**. 

Y 250 USD equivalen a **1.025.000 COP** (según la tasa de cambio actual de 4.100 COP por dólar).


  [Ollama] Gracias por la pregunta! En Tunja, el clima actual es de 14°C con lluvia.

Y para convertir 250 USD a pesos colombianos (COP), el resultado es aproximadamente 1025000.0 COP.


**Cuando una herramienta falla.** Las herramientas reales fallan: una cuenta que no existe, un servicio caído, un argumento inválido. Lo importante es que el error **no rompa el programa**, sino que **llegue al modelo como un `ToolMessage` con `status="error"`**, para que pueda explicárselo al usuario o corregir su llamada. El bucle de arriba ya lo hace con `try/except`. Lo probamos con una herramienta que falla para cuentas desconocidas:

In [17]:
@tool
def consultar_saldo(cuenta: str) -> str:
    """Consulta el saldo de una cuenta bancaria por su número de cuenta."""
    if cuenta != "001":
        raise ValueError(f"La cuenta {cuenta} no existe")
    return "Saldo: $120.000"


mostrar(ejecutar_con_herramientas(modelo, [consultar_saldo], "¿Cuál es el saldo de la cuenta 999?"))

usuario      | ¿Cuál es el saldo de la cuenta 999?
modelo       | pide: consultar_saldo({'cuenta': '999'})
herramienta  | consultar_saldo -> Error: ValueError: La cuenta 999 no existe  [ERROR]
modelo       | La cuenta 999 no existe. ¿Podrías verificar el número de cuenta o proporcionarme otro número para consultar el saldo?


**Controlar la elección de herramientas.** Por defecto el modelo decide libremente si usar una herramienta (`tool_choice="auto"`). `bind_tools` permite forzarlo: `tool_choice="any"` obliga a usar alguna herramienta y `tool_choice="nombre"` obliga a usar una concreta (útil, por ejemplo, para extraer datos estructurados). Pero **los parámetros no son idénticos entre proveedores**: `parallel_tool_calls=False` (impedir llamadas paralelas) solo existe en OpenAI. Probamos las tres variantes con cada proveedor:

In [18]:
filas = []
for proveedor, m in MODELOS.items():
    for etiqueta, argumentos, texto in [
        ("tool_choice='calculadora'", {"tool_choice": "calculadora"}, "Hola, ¿cómo estás?"),
        ("tool_choice='any'", {"tool_choice": "any"}, "Hola, ¿cómo estás?"),
        ("parallel_tool_calls=False", {"parallel_tool_calls": False}, "Dime el clima de Cali y cuánto es 2+2."),
    ]:
        try:
            r = m.bind_tools(HERRAMIENTAS_BASE, **argumentos).invoke(texto)
            resultado = f"llamadas: {[c['name'] for c in r.tool_calls]}"
        except Exception as error:
            resultado = f"no soportado ({type(error).__name__})"
        filas.append({"proveedor": proveedor, "opción": etiqueta, "resultado": resultado})
pd.DataFrame(filas).pivot(index="opción", columns="proveedor", values="resultado")

proveedor,Google,Ollama,OpenAI
opción,,,
parallel_tool_calls=False,no soportado (ValidationError),no soportado (TypeError),llamadas: ['clima']
tool_choice='any',llamadas: ['clima'],llamadas: ['calculadora'],llamadas: ['clima']
tool_choice='calculadora',llamadas: ['calculadora'],llamadas: ['calculadora'],llamadas: ['calculadora']


La abstracción de LangChain unifica lo común, pero las opciones específicas de cada proveedor **no siempre se trasladan** (aquí, `parallel_tool_calls` en Google y Ollama). Conviene tenerlo presente cuando un código "portable" falla al cambiar de proveedor.

**¿Qué tanto importa la descripción de la herramienta?** Dijimos que el modelo solo ve nombre, descripción y esquema. Lo comprobamos con datos: 10 preguntas, cada una con la herramienta que *debería* usarse (o ninguna: una pregunta de cultura general y un poema). Comparamos dos versiones de las mismas cuatro herramientas: las **bien descritas** de arriba y unas **vagas** (nombres `f1` a `f4` y la descripción *"Hace algo útil."*, con los mismos argumentos). Medimos cuántas veces el modelo elige exactamente el conjunto de herramientas correcto, **sin ejecutar** nada.

In [19]:
def vaga(herramienta, nombre):
    """La misma función con un nombre opaco y una descripción inútil (el esquema se infiere de las anotaciones)."""
    return StructuredTool.from_function(func=herramienta.func, name=nombre, description="Hace algo útil.")


VAGAS = [vaga(h, f"f{i}") for i, h in enumerate(HERRAMIENTAS_BASE, start=1)]
EQUIVALENCIA = {f"f{i}": h.name for i, h in enumerate(HERRAMIENTAS_BASE, start=1)}

CASOS = [
    ("¿Qué clima hace en Cali?", {"clima"}),
    ("¿Cuántos euros son 500000 pesos?", {"convertir_moneda"}),
    ("¿Cuánto es 37 por 48?", {"calculadora"}),
    ("¿Cuántas personas hay por grupo en el proyecto del módulo 4 del curso?", {"buscar_documentos"}),
    ("¿Cuál es la capital de Francia?", set()),
    ("Voy a viajar a Bogotá mañana, ¿llevo paraguas?", {"clima"}),
    ("¿Cuántos pesos son 360 dólares?", {"convertir_moneda"}),
    ("¿Cuántos módulos tiene el curso?", {"buscar_documentos"}),
    ("Escribe un poema de dos versos sobre la lluvia.", set()),
    ("¿Cuánto es el 15 % de 2400?", {"calculadora"}),
]


def herramientas_elegidas(m, herramientas, pregunta):
    respuesta = m.bind_tools(herramientas).invoke(pregunta)
    return {EQUIVALENCIA.get(c["name"], c["name"]) for c in respuesta.tool_calls}


filas, fallos = [], []
for proveedor, m in MODELOS.items():
    for nombre, herramientas in [("bien descritas", HERRAMIENTAS_BASE), ("vagas", VAGAS)]:
        with ThreadPoolExecutor(4) as pool:
            elegidas = list(pool.map(lambda c: herramientas_elegidas(m, herramientas, c[0]), CASOS))
        aciertos = [e == esperado for e, (_, esperado) in zip(elegidas, CASOS)]
        filas.append({"proveedor": proveedor, "descripciones": nombre, "aciertos (de 10)": sum(aciertos)})
        fallos += [{"proveedor": proveedor, "descripciones": nombre, "pregunta": CASOS[i][0], "esperado": sorted(CASOS[i][1]) or "ninguna",
                    "eligió": sorted(elegidas[i]) or "ninguna"} for i, ok in enumerate(aciertos) if not ok]
display(pd.DataFrame(filas).pivot(index="proveedor", columns="descripciones", values="aciertos (de 10)"))
print("Casos en los que el modelo eligió mal:")
display(pd.DataFrame(fallos).set_index(["proveedor", "descripciones"]))

descripciones,bien descritas,vagas
proveedor,,
Google,10,8
Ollama,7,2
OpenAI,10,9


Casos en los que el modelo eligió mal:


pregunta  \
proveedor descripciones                                                                            
OpenAI    vagas                                                 ¿Cuántos módulos tiene el curso?   
Google    vagas                                                  ¿Cuántos pesos son 360 dólares?   
          vagas                                                 ¿Cuántos módulos tiene el curso?   
Ollama    bien descritas  ¿Cuántas personas hay por grupo en el proyecto del módulo 4 del curso?   
          bien descritas                                         ¿Cuál es la capital de Francia?   
          bien descritas                         Escribe un poema de dos versos sobre la lluvia.   
          vagas                                                         ¿Qué clima hace en Cali?   
          vagas                                                            ¿Cuánto es 37 por 48?   
          vagas           ¿Cuántas personas hay por grupo en el proyecto del módulo 4 del curso?   
          vagas                                                  ¿Cuál es la capital de Francia?   
          vagas                                   Voy a viajar a Bogotá mañana, ¿llevo paraguas?   
          vagas                                                 ¿Cuántos módulos tiene el curso?   
          vagas                                  Escribe un poema de dos versos sobre la lluvia.   
          vagas                                                      ¿Cuánto es el 15 % de 2400?   

                                     esperado               eligió  
proveedor descripciones                                             
OpenAI    vagas           [buscar_documentos]              ninguna  
Google    vagas            [convertir_moneda]  [buscar_documentos]  
          vagas           [buscar_documentos]              ninguna  
Ollama    bien descritas  [buscar_documentos]        [calculadora]  
          bien descritas              ninguna  [buscar_documentos]  
          bien descritas              ninguna        [calculadora]  
          vagas                       [clima]        [calculadora]  
          vagas                 [calculadora]   [convertir_moneda]  
          vagas           [buscar_documentos]        [calculadora]  
          vagas                       ninguna              [clima]  
          vagas                       [clima]        [calculadora]  
          vagas           [buscar_documentos]        [calculadora]  
          vagas                       ninguna        [calculadora]  
          vagas                 [calculadora]   [convertir_moneda]

Las descripciones importan, y **más cuanto más pequeño es el modelo**:

| | OpenAI | Google | Ollama (3B) |
|---|---|---|---|
| Herramientas bien descritas | 10/10 | 10/10 | 7/10 |
| Herramientas vagas | 9/10 | 8/10 | 2/10 |

Con los modelos grandes la caída es pequeña porque los **nombres y tipos de los argumentos** (`ciudad`, `monto`, `origen`) todavía dan pistas; los fallos se concentran en las preguntas cuya herramienta no se puede deducir de los argumentos, como la de los documentos del curso. El modelo local casi pierde el rumbo con las descripciones vagas (2 de 10) y falló además con descripciones buenas, en tres preguntas; en dos de ellas (la capital de Francia y el poema) **llamó una herramienta que no hacía falta**. Con 10 preguntas por celda los números son orientativos y cambian algo entre ejecuciones, pero la dirección es clara: la descripción es parte del sistema, y hay que escribirla y probarla como un prompt.

**Buenas prácticas con herramientas:**

| Práctica | Por qué |
|---|---|
| **Nombre y docstring claros**, con cuándo usarla y cuándo no | Es lo único que el modelo sabe de la herramienta |
| **Pocos argumentos, tipados**, con `Literal` o enumeraciones cuando los valores están acotados | Menos errores de formato y validación previa |
| **Pocas herramientas por agente** (decenas degradan la elección) | Cada descripción ocupa contexto y compite con las demás |
| **Principio de mínimo privilegio** | Una herramienta que borra archivos o envía correos tiene que ser indispensable y, si es sensible, pedir aprobación humana |
| **Validar los argumentos** y no ejecutar nunca lo que el modelo escribe (`eval`, SQL, comandos de shell) | El modelo, o quien lo manipule con inyección de prompts, controla los argumentos |
| **Devolver errores claros** al modelo, sin exponer detalles internos | Permite que se recupere sin filtrar información sensible |
| **Que sean idempotentes** o tengan una confirmación | Un reintento no debe cobrar dos veces |
| **Limitar el tamaño de la salida** | Una respuesta enorme llena el contexto y encarece cada paso siguiente |

<a id="rag-con-langchain"></a>
<h2 style="color: #007ACC;">RAG con LangChain</h2>

La **generación aumentada con recuperación** (RAG; Lewis et al., 2020) responde con base en documentos que el modelo no memorizó: se **recuperan** los fragmentos relevantes para la pregunta y se incluyen en el prompt. LangChain tiene un componente para cada etapa:

```text
 documentos ─► dividir en fragmentos ─► embeddings ─► almacén vectorial          (indexación, una vez)
 pregunta ──► embedding ──► fragmentos más similares ──► prompt + modelo ──► respuesta   (consulta)
```

- **`Document`**: un texto con **metadatos** (`page_content` y `metadata`).
- **Divisores de texto** (`RecursiveCharacterTextSplitter`): parten un documento largo en fragmentos de un tamaño dado, con **solapamiento** para no cortar ideas por la mitad.
- **Embeddings** (`OpenAIEmbeddings`, `GoogleGenerativeAIEmbeddings`, `OllamaEmbeddings`): convierten texto en vectores (cuaderno 12).
- **Almacén vectorial** (`InMemoryVectorStore` aquí; hay integraciones con Chroma, FAISS, pgvector y muchos más): guarda los vectores y busca por similitud.
- **Recuperador** (`retriever`): un *Runnable* que recibe una pregunta y devuelve documentos; se obtiene con `almacen.as_retriever()`.

Antes teníamos una búsqueda por **coincidencia de palabras**; la reemplazamos por una **búsqueda semántica**. Usamos el corpus del curso de las herramientas y el proveedor de embeddings que tengas disponible.

In [20]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter


def crear_embeddings():
    """Embeddings del primer proveedor disponible que los ofrezca (Ollama requiere `ollama pull nomic-embed-text`)."""
    if "OpenAI" in DISPONIBLES:
        from langchain_openai import OpenAIEmbeddings
        return "OpenAI", OpenAIEmbeddings(model="text-embedding-3-small")
    if "Google" in DISPONIBLES:
        from langchain_google_genai import GoogleGenerativeAIEmbeddings
        return "Google", GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
    from langchain_ollama import OllamaEmbeddings
    return "Ollama", OllamaEmbeddings(model="nomic-embed-text")


proveedor_embeddings, embeddings = crear_embeddings()

# 1. dividir el documento en fragmentos con solapamiento
documento = Document(page_content="\n\n".join(PARRAFOS), metadata={"fuente": "programa_del_curso"})
divisor = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=50)
fragmentos = divisor.split_documents([documento])
print(f"1 documento de {len(documento.page_content)} caracteres -> {len(fragmentos)} fragmentos de {min(len(f.page_content) for f in fragmentos)} a {max(len(f.page_content) for f in fragmentos)} caracteres")

# 2. indexar: embeddings + almacén vectorial
almacen = InMemoryVectorStore.from_documents(fragmentos, embeddings)
recuperador = almacen.as_retriever(search_kwargs={"k": 2})

# 3. consultar: los fragmentos más similares, con su puntaje de similitud
for consulta, k in [("¿cuánta gente forma cada equipo del proyecto final?", 2), ("¿qué se estudia en las semanas 5 y 6?", 6)]:
    print(f"\nPregunta: {consulta}")
    for puesto, (doc, puntaje) in enumerate(almacen.similarity_search_with_score(consulta, k=k), start=1):
        texto = doc.page_content.replace("\n\n", " ⏎ ")
        print(f"  {puesto}. {puntaje:.3f} | {texto if puesto == 1 else texto[:70] + '...'}")   # el primero completo; los demás, recortados

1 documento de 1591 caracteres -> 6 fragmentos de 171 a 330 caracteres



Pregunta: ¿cuánta gente forma cada equipo del proyecto final?


  1. 0.414 | El proyecto del módulo 4 se desarrolla en grupos de 2 a 3 personas y debe poder ejecutarse en Google Colab con una GPU T4. ⏎ El proyecto del módulo 4 exige una presentación de 10 minutos con los hallazgos principales y una auditoría ética con su tarjeta del modelo.
  2. 0.348 | Módulo 4 (semanas 7 y 8): atención y modelos preentrenados: Transforme...

Pregunta: ¿qué se estudia en las semanas 5 y 6?


  1. 0.500 | Módulo 2 (semanas 3 y 4): métodos probabilísticos: Bag-of-Words y N-gramas, clasificación con NLTK, modelos ocultos de Markov, algoritmo de Viterbi y modelos de Markov de máxima entropía (MEMM). ⏎ Módulo 3 (semanas 5 y 6): modelos secuenciales: embeddings con redes neuronales, Word2Vec y GloVe, RNN, LSTM y GRU, y modelos Seq2Seq.
  2. 0.440 | Módulo 4 (semanas 7 y 8): atención y modelos preentrenados: Transforme...
  3. 0.438 | Módulo 1 (semanas 1 y 2): introducción al PLN, preprocesamiento de tex...
  4. 0.337 | El curso de Procesamiento de Lenguaje Natural pertenece a la Especiali...
  5. 0.333 | Los requisitos de acceso son Aprendizaje Automático, Ciencia de Datos ...
  6. 0.328 | El proyecto del módulo 4 se desarrolla en grupos de 2 a 3 personas y d...


Fíjate en la primera pregunta: dice *"equipo"* y el documento dice *"grupos"*. Una búsqueda por palabras no los conectaría; la búsqueda **semántica** sí.

Y mira la segunda: el fragmento que quedó primero es el correcto, porque **contiene el módulo 3** (semanas 5 y 6). Pero también contiene el módulo 2: el divisor unió dos párrafos vecinos hasta llegar a los 350 caracteres. El **tamaño de los fragmentos** es un compromiso: fragmentos pequeños son más precisos pero pueden separar una idea de su contexto; fragmentos grandes conservan el contexto pero traen texto que no se pidió. Observa además los puntajes: el segundo y el tercer fragmento (0,44) están muy cerca del primero (0,50). Los embeddings capturan sobre todo el **tema**, no las cifras exactas: *"semanas 5 y 6"* se parece mucho a *"semanas 3 y 4"*. Aquí el corpus es diminuto y funcionó, pero en uno grande es un riesgo conocido de la búsqueda puramente semántica; se mitiga con **más fragmentos recuperados** (`k` mayor), **búsqueda híbrida** (palabras clave más embeddings), **filtros por metadatos** o un **reordenamiento** (*reranking*) de los resultados. Ahora armamos la **cadena RAG** con LCEL: el diccionario inicial ejecuta en paralelo el recuperador (para el contexto) y deja pasar la pregunta. El prompt incluye una **salida de escape** (*"si no está en el contexto, di que no lo sabes"*) para que el modelo no invente cuando los documentos no contienen la respuesta.

In [21]:
def formatear(documentos):
    return "\n\n".join(d.page_content for d in documentos)


prompt_rag = ChatPromptTemplate.from_messages([
    ("system", "Responde usando SOLO el contexto. Si la respuesta no está en el contexto, responde exactamente: «No lo sé según los documentos»."),
    ("human", "Contexto:\n{contexto}\n\nPregunta: {pregunta}"),
])
cadena_rag = {"contexto": recuperador | formatear, "pregunta": RunnablePassthrough()} | prompt_rag | modelo | StrOutputParser()

for pregunta in ["¿Cuántas personas hay por grupo en el proyecto del módulo 4?", "¿En qué semanas se estudian los Transformers?",
                 "¿Qué se estudia en el módulo 2?", "¿Cuál es la fecha del examen final?"]:
    print(f"{pregunta}\n  -> {cadena_rag.invoke(pregunta).strip()}\n")

¿Cuántas personas hay por grupo en el proyecto del módulo 4?
  -> En el proyecto del módulo 4, los grupos están formados por 2 a 3 personas.



¿En qué semanas se estudian los Transformers?
  -> Los Transformers se estudian en las semanas 7 y 8, dentro del Módulo 4.



¿Qué se estudia en el módulo 2?
  -> En el módulo 2 (semanas 3 y 4) se estudian métodos probabilísticos: Bag-of-Words y N-gramas, clasificación con NLTK, modelos ocultos de Markov, algoritmo de Viterbi y modelos de Markov de máxima entropía (MEMM).



¿Cuál es la fecha del examen final?
  -> No lo sé según los documentos.



Las tres primeras preguntas se responden con el contexto; la cuarta **no está en los documentos** y el modelo se abstiene en lugar de inventar una fecha.

Este RAG tiene una forma **fija**: siempre recupera y siempre responde. Un enfoque más flexible, el **RAG agéntico**, le da al modelo el recuperador como una **herramienta** y deja que él decida *si* buscar, *qué* buscar y *cuántas veces*. Redefinimos `buscar_documentos` con el mismo nombre y la misma descripción, pero con búsqueda semántica; la usaremos en los agentes.

Este RAG es deliberadamente mínimo. En el cuaderno 20 lo estudiamos a fondo: estrategias de *chunking*, de recuperación (búsqueda híbrida, *reranking*, transformación de consultas) y distintos tipos de RAG (correctivo, adaptativo, agéntico, GraphRAG).

In [22]:
@tool
def buscar_documentos(consulta: str) -> str:
    """Busca información en los documentos del curso de PLN: programa, módulos, semanas, proyectos y requisitos."""
    return formatear(recuperador.invoke(consulta)) or "Sin resultados."


HERRAMIENTAS = [clima, convertir_moneda, calculadora, buscar_documentos]  # las que usarán los agentes
print(buscar_documentos.invoke({"consulta": "presentación del proyecto"}))

El proyecto del módulo 4 se desarrolla en grupos de 2 a 3 personas y debe poder ejecutarse en Google Colab con una GPU T4.

El proyecto del módulo 4 exige una presentación de 10 minutos con los hallazgos principales y una auditoría ética con su tarjeta del modelo.

Módulo 4 (semanas 7 y 8): atención y modelos preentrenados: Transformers, BERT y GPT con fine-tuning, aplicaciones de modelos preentrenados, prompt engineering, LangChain y LangGraph, y ética en PLN.

Cada módulo tiene cuadernos de clase, talleres prácticos con su solución y un proyecto de aplicación.


<a id="langgraph-fundamentos"></a>
<h2 style="color: #007ACC;">LangGraph: fundamentos</h2>

Una cadena LCEL es un **flujo lineal** (o con ramas) que va de la entrada a la salida **una sola vez**. Muchas aplicaciones necesitan más:

- **Ciclos**: repetir un paso hasta que se cumpla una condición (un agente que llama herramientas hasta tener la respuesta; un redactor que corrige hasta pasar una revisión).
- **Decisiones**: elegir el siguiente paso según el resultado del anterior.
- **Estado**: recordar lo ocurrido a lo largo de los pasos y de los turnos de una conversación.
- **Pausas**: detenerse a esperar la aprobación de una persona y retomar después.

**LangGraph** modela la aplicación como un **grafo** con cuatro elementos:

| Elemento | Qué es |
|---|---|
| **Estado** (*state*) | La memoria compartida del grafo: un diccionario tipado (`TypedDict` o Pydantic) que todos los nodos leen y actualizan |
| **Nodos** (*nodes*) | Funciones de Python que reciben el estado y devuelven **las actualizaciones** que quieren hacer (no el estado completo) |
| **Aristas** (*edges*) | Las transiciones entre nodos: fijas, o **condicionales** (una función decide el siguiente nodo) |
| **Reductores** (*reducers*) | La regla para combinar una actualización con el valor actual de una clave: por defecto la **reemplaza**; con `operator.add` la **acumula**; con `add_messages` agrega mensajes |

Se construye con `StateGraph`, se añaden nodos y aristas desde el nodo especial `START` hasta `END`, y se **compila** con `compile()`, lo que devuelve un *Runnable* (con `invoke`, `stream`, etc.). Por dentro, la ejecución avanza en **pasos**: en cada paso se ejecutan los nodos activos, se aplican sus actualizaciones al estado con los reductores y se calculan los nodos siguientes.

Empezamos con un grafo **sin LLM**, para ver la mecánica sin distracciones: un nodo que duplica un número, otro que lo anota y una arista condicional que vuelve a duplicar (un ciclo) hasta pasar de 100.

In [23]:
from langgraph.graph import StateGraph, START, END, MessagesState


class EstadoContador(TypedDict):
    numero: int
    pasos: Annotated[list[str], operator.add]  # reductor: las listas se concatenan en lugar de reemplazarse


def duplicar(estado: EstadoContador):
    return {"numero": estado["numero"] * 2}                        # un nodo devuelve solo lo que cambia


def anotar(estado: EstadoContador):
    return {"pasos": [f"número = {estado['numero']}"]}


def seguir(estado: EstadoContador) -> Literal["duplicar", "__end__"]:
    return "duplicar" if estado["numero"] < 100 else END           # arista condicional: decide el siguiente nodo


constructor = StateGraph(EstadoContador)
constructor.add_node("duplicar", duplicar)
constructor.add_node("anotar", anotar)
constructor.add_edge(START, "duplicar")
constructor.add_edge("duplicar", "anotar")
constructor.add_conditional_edges("anotar", seguir)
contador = constructor.compile()

print("invoke:", contador.invoke({"numero": 20, "pasos": []}))
print("\nEl grafo (un ciclo: se vuelve a duplicar hasta pasar de 100):")
contador.get_graph().print_ascii()

invoke: {'numero': 160, 'pasos': ['número = 40', 'número = 80', 'número = 160']}

El grafo (un ciclo: se vuelve a duplicar hasta pasar de 100):
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+----------+   
| duplicar |   
+----------+   
      .        
      .        
      .        
  +--------+   
  | anotar |   
  +--------+   
      .        
      .        
      .        
 +---------+   
 | __end__ |   
 +---------+   


El estado final conserva la lista `pasos` completa porque su reductor es `operator.add`; `numero`, sin reductor, se **reemplaza** en cada paso. Cada nodo devuelve solo lo que cambia: `duplicar` actualiza `numero` y `anotar` añade a `pasos`. El grafo también se puede exportar como texto en formato **Mermaid** (`get_graph().draw_mermaid()`), que se pega en [mermaid.live](https://mermaid.live) para obtener una imagen. (`draw_mermaid_png()` la genera directamente, pero lo hace llamando a un servicio web externo.)

**Streaming del estado.** En lugar de esperar el resultado final, `stream` emite lo que ocurre en cada paso. El parámetro `stream_mode` elige qué: `"updates"` emite solo **lo que cambió** en cada nodo; `"values"`, el **estado completo** tras cada paso.

In [24]:
print("stream_mode='updates' (solo los cambios de cada nodo):")
for evento in contador.stream({"numero": 20, "pasos": []}, stream_mode="updates"):
    print("  ", evento)

print("\nstream_mode='values' (el estado completo después de cada paso):")
for estado_completo in contador.stream({"numero": 20, "pasos": []}, stream_mode="values"):
    print("  ", estado_completo)

stream_mode='updates' (solo los cambios de cada nodo):
   {'duplicar': {'numero': 40}}
   {'anotar': {'pasos': ['número = 40']}}
   {'duplicar': {'numero': 80}}
   {'anotar': {'pasos': ['número = 80']}}
   {'duplicar': {'numero': 160}}
   {'anotar': {'pasos': ['número = 160']}}

stream_mode='values' (el estado completo después de cada paso):
   {'numero': 20, 'pasos': []}
   {'numero': 40, 'pasos': []}
   {'numero': 40, 'pasos': ['número = 40']}
   {'numero': 80, 'pasos': ['número = 40']}
   {'numero': 80, 'pasos': ['número = 40', 'número = 80']}
   {'numero': 160, 'pasos': ['número = 40', 'número = 80']}
   {'numero': 160, 'pasos': ['número = 40', 'número = 80', 'número = 160']}


**El reductor de mensajes.** Para las conversaciones, LangGraph incluye `MessagesState`, un estado con una sola clave, `messages`, cuyo reductor es `add_messages`. Este reductor **agrega** los mensajes nuevos a la lista, y si llega un mensaje con un `id` que ya existe, **lo reemplaza** en lugar de duplicarlo. Es el estado que usan todos los agentes:

In [25]:
from langgraph.graph.message import add_messages

historial = [HumanMessage("hola", id="1")]
print("agregar un mensaje nuevo :", [(type(m).__name__, m.text) for m in add_messages(historial, [AIMessage("¿qué tal?", id="2")])])
print("mismo id: lo reemplaza   :", [(type(m).__name__, m.text) for m in add_messages(historial, [HumanMessage("hola, corregido", id="1")])])

agregar un mensaje nuevo : [('HumanMessage', 'hola'), ('AIMessage', '¿qué tal?')]
mismo id: lo reemplaza   : [('HumanMessage', 'hola, corregido')]


<a id="workflows-con-langgraph"></a>
<h2 style="color: #007ACC;">Workflows con LangGraph</h2>

Antes de llegar a los agentes conviene distinguir dos formas de organizar una aplicación con LLM (Anthropic, 2024):

- Un **workflow** tiene un camino **definido por el programador**: los LLM y las herramientas se orquestan con código fijo (cadenas, ramas, ciclos con condición de parada explícita). Es más predecible, barato y fácil de depurar.
- Un **agente** deja que el **LLM decida** el camino: qué herramientas llamar, en qué orden y cuándo parar. Es más flexible, pero menos predecible y más caro.

La recomendación habitual es usar el esquema **más simple que resuelva el problema**: un solo prompt, luego un workflow y, solo si hace falta, un agente. Con LangGraph, un workflow se escribe igual que un agente; solo cambia quién decide las transiciones. Veamos tres patrones con LLM reales.

**1. Enrutamiento.** Un primer nodo clasifica la entrada (con salida estructurada) y una arista condicional la manda al nodo especializado.

In [26]:
class Ruta(BaseModel):
    categoria: Literal["tecnica", "administrativa", "otra"] = Field(
        description="tecnica: conceptos de PLN o IA; administrativa: organización del curso; otra: cualquier otro tema")


class EstadoRuta(TypedDict):
    pregunta: str
    categoria: str
    respuesta: str


clasificador_ruta = modelo.with_structured_output(Ruta)


def clasificar(estado: EstadoRuta):
    return {"categoria": clasificador_ruta.invoke(f"Clasifica la pregunta de un estudiante: {estado['pregunta']}").categoria}


def responder_tecnica(estado: EstadoRuta):
    return {"respuesta": modelo.invoke(f"Eres profesor de PLN. Responde en dos frases: {estado['pregunta']}").text.strip()}


def responder_administrativa(estado: EstadoRuta):
    contexto = buscar_documentos.invoke({"consulta": estado["pregunta"]})  # este nodo usa el recuperador
    return {"respuesta": modelo.invoke(f"Responde solo con este contexto:\n{contexto}\n\nPregunta: {estado['pregunta']}").text.strip()}


def rechazar(estado: EstadoRuta):
    return {"respuesta": "Solo puedo ayudar con preguntas del curso."}


enrutador = StateGraph(EstadoRuta)
enrutador.add_node("clasificar", clasificar)
enrutador.add_node("responder_tecnica", responder_tecnica)
enrutador.add_node("responder_administrativa", responder_administrativa)
enrutador.add_node("rechazar", rechazar)
enrutador.add_edge(START, "clasificar")
enrutador.add_conditional_edges("clasificar", lambda e: e["categoria"],
                                {"tecnica": "responder_tecnica", "administrativa": "responder_administrativa", "otra": "rechazar"})
for nodo in ["responder_tecnica", "responder_administrativa", "rechazar"]:
    enrutador.add_edge(nodo, END)
app_ruta = enrutador.compile()

for pregunta in ["¿Qué diferencia hay entre una LSTM y una GRU?", "¿Cuántas personas hay por grupo en el proyecto del módulo 4?",
                 "¿Quién ganó el mundial de fútbol de 2014?"]:
    r = app_ruta.invoke({"pregunta": pregunta})
    print(f"{pregunta}\n  [{r['categoria']}] {r['respuesta']}\n")
app_ruta.get_graph().print_ascii()

¿Qué diferencia hay entre una LSTM y una GRU?
  [tecnica] La principal diferencia entre una LSTM y una GRU es que la LSTM tiene tres puertas (entrada, olvido y salida) y una celda de memoria separada, mientras que la GRU combina las puertas de entrada y olvido en una sola puerta de actualización y no tiene una celda de memoria separada. Esto hace que las GRU sean más simples y computacionalmente eficientes, aunque ambas son efectivas para capturar dependencias a largo plazo en secuencias.



¿Cuántas personas hay por grupo en el proyecto del módulo 4?
  [administrativa] En el proyecto del módulo 4, los grupos deben estar conformados por 2 a 3 personas.



¿Quién ganó el mundial de fútbol de 2014?
  [otra] Solo puedo ayudar con preguntas del curso.

                                 +-----------+                                       
                                 | __start__ |                                       
                                 +-----------+                                       
                                       *                                             
                                       *                                             
                                       *                                             
                                +------------+                                       
                                | clasificar |..                                     
                          ......+------------+  ......                               
                     .....            .               .....                          
               ......                .       

**2. Paralelización.** Cuando dos pasos no dependen entre sí, se lanzan **a la vez** desde el mismo nodo de origen y se **unen** en un nodo posterior. LangGraph ejecuta las ramas en paralelo dentro del mismo paso. Comparamos el tiempo con ejecutarlas una tras otra:

In [27]:
class EstadoParalelo(TypedDict):
    texto: str
    resumen: str
    traduccion: str
    informe: str


def resumir(estado: EstadoParalelo):
    return {"resumen": modelo.invoke(f"Resume en una frase: {estado['texto']}").text.strip()}


def traducir_a_ingles(estado: EstadoParalelo):
    return {"traduccion": modelo.invoke(f"Traduce al inglés, solo la traducción: {estado['texto']}").text.strip()}


def combinar(estado: EstadoParalelo):
    return {"informe": f"RESUMEN: {estado['resumen']}\nTRADUCCIÓN: {estado['traduccion']}"}


paralelo = StateGraph(EstadoParalelo)
paralelo.add_node("resumir", resumir)
paralelo.add_node("traducir", traducir_a_ingles)
paralelo.add_node("combinar", combinar)
paralelo.add_edge(START, "resumir")      # dos aristas desde START: las dos ramas corren a la vez
paralelo.add_edge(START, "traducir")
paralelo.add_edge("resumir", "combinar")  # combinar espera a que terminen las dos
paralelo.add_edge("traducir", "combinar")
app_paralelo = paralelo.compile()

texto = ("LangGraph modela una aplicación como un grafo con estado. Cada nodo es una función que actualiza el estado compartido "
         "y las aristas deciden qué nodo se ejecuta después, lo que permite ciclos, memoria y pausas para intervención humana.")
t = time.perf_counter(); r = app_paralelo.invoke({"texto": texto}); t_grafo = time.perf_counter() - t
t = time.perf_counter(); resumir({"texto": texto}); traducir_a_ingles({"texto": texto}); t_secuencial = time.perf_counter() - t
print(r["informe"])
print(f"\ncon el grafo (ramas en paralelo): {t_grafo:.1f} s | una rama tras otra: {t_secuencial:.1f} s")

RESUMEN: LangGraph representa una aplicación como un grafo de funciones que actualizan un estado compartido, donde las aristas controlan la ejecución secuencial, permitiendo ciclos, memoria y pausas para intervención humana.
TRADUCCIÓN: LangGraph models an application as a state graph. Each node is a function that updates the shared state, and the edges determine which node executes next, allowing cycles, memory, and pauses for human intervention.

con el grafo (ramas en paralelo): 1.2 s | una rama tras otra: 1.6 s


**3. Evaluador-optimizador (reflexión).** Un nodo genera un borrador, otro lo **verifica** y, si no cumple, la arista condicional **vuelve** al primero con las observaciones. Es un ciclo con **condición de parada explícita** y un **límite de iteraciones**: nunca se deja un ciclo sin tope.

Aquí el verificador es **código determinista**, no otro LLM: contar palabras es algo que los LLM hacen mal y el código hace bien. Pedimos una descripción de producto de **exactamente 12 palabras**:

In [28]:
class EstadoRedaccion(TypedDict):
    producto: str
    borrador: str
    observaciones: str
    iteraciones: int
    historial: Annotated[list[str], operator.add]


PALABRAS_OBJETIVO = 12


def redactar(estado: EstadoRedaccion):
    instruccion = f"Escribe una descripción de producto de exactamente {PALABRAS_OBJETIVO} palabras para: {estado['producto']}. Responde solo con la descripción."
    if estado["observaciones"]:
        instruccion += f"\nTu borrador anterior fue rechazado: {estado['observaciones']} Corrígelo."
    borrador = modelo.invoke(instruccion).text.strip()
    return {"borrador": borrador, "iteraciones": estado["iteraciones"] + 1, "historial": [f"{len(borrador.split())} palabras: {borrador}"]}


def verificar(estado: EstadoRedaccion):
    n = len(estado["borrador"].split())
    return {"observaciones": "" if n == PALABRAS_OBJETIVO else f"Tiene {n} palabras y debe tener exactamente {PALABRAS_OBJETIVO}."}


def decidir(estado: EstadoRedaccion) -> Literal["redactar", "__end__"]:
    return END if not estado["observaciones"] or estado["iteraciones"] >= 4 else "redactar"   # parada: cumple o se acaban los intentos


reflexion = StateGraph(EstadoRedaccion)
reflexion.add_node("redactar", redactar)
reflexion.add_node("verificar", verificar)
reflexion.add_edge(START, "redactar")
reflexion.add_edge("redactar", "verificar")
reflexion.add_conditional_edges("verificar", decidir)
app_reflexion = reflexion.compile()

r = app_reflexion.invoke({"producto": "una ruana de lana de Boyacá", "borrador": "", "observaciones": "", "iteraciones": 0, "historial": []})
print(f"Iteraciones: {r['iteraciones']}")
for linea in r["historial"]:
    print("  -", linea)
print("Resultado final:", "cumple" if not r["observaciones"] else f"no cumple ({r['observaciones']})")

Iteraciones: 2
  - 11 palabras: Ruana artesanal de lana, cálida y tradicional, perfecta para clima frío.
  - 12 palabras: Ruana de lana artesanal de Boyacá, cálida, elegante y perfecta para invierno.
Resultado final: cumple


En esta ejecución el primer borrador tuvo 11 palabras y el verificador lo rechazó; el segundo tuvo exactamente 12. En otras ejecuciones el modelo necesitó hasta tres intentos. Es la razón de que el verificador sea **código**: el modelo falla de forma repetida en una tarea tan simple como contar palabras, y un ciclo con una comprobación determinista y un tope de iteraciones lo corrige de forma fiable.

<a id="persistencia-intervencion-humana-y-streaming"></a>
<h2 style="color: #007ACC;">Persistencia, intervención humana y streaming</h2>

**Memoria con *checkpointers*.** Por defecto, cada `invoke` empieza con un estado nuevo: el grafo **no recuerda** nada entre llamadas. Al compilar con un ***checkpointer*** (por ejemplo `InMemorySaver`), LangGraph **guarda una instantánea del estado después de cada paso**, asociada a un **hilo** (`thread_id`). Si invocas de nuevo con el mismo `thread_id`, el grafo continúa desde donde quedó; con otro `thread_id`, es una conversación independiente.

`InMemorySaver` guarda en memoria (se pierde al cerrar el programa). Para persistir entre ejecuciones hay checkpointers de SQLite y de PostgreSQL (paquetes `langgraph-checkpoint-sqlite` y `langgraph-checkpoint-postgres`) con la misma interfaz. Probamos un chatbot sin memoria y con memoria:

In [29]:
from langgraph.checkpoint.memory import InMemorySaver


def conversar(estado: MessagesState):
    return {"messages": [modelo.invoke([SystemMessage("Eres un asistente breve.")] + estado["messages"])]}


chat = StateGraph(MessagesState)
chat.add_node("conversar", conversar)
chat.add_edge(START, "conversar")
chat.add_edge("conversar", END)
sin_memoria = chat.compile()
con_memoria = chat.compile(checkpointer=InMemorySaver())


def decir(app, texto, hilo=None):
    configuracion = {"configurable": {"thread_id": hilo}} if hilo else None
    return app.invoke({"messages": [HumanMessage(texto)]}, configuracion)["messages"][-1].text.strip()


print("SIN memoria")
decir(sin_memoria, "Me llamo Laura.")
print("  ¿Cómo me llamo? ->", decir(sin_memoria, "¿Cómo me llamo?"))

print("\nCON memoria, hilo 'ana' y hilo 'luis'")
decir(con_memoria, "Me llamo Laura.", "ana")
decir(con_memoria, "Me llamo Pedro.", "luis")
print("  hilo ana  -> ¿Cómo me llamo?:", decir(con_memoria, "¿Cómo me llamo?", "ana"))
print("  hilo luis -> ¿Cómo me llamo?:", decir(con_memoria, "¿Cómo me llamo?", "luis"))

SIN memoria


  ¿Cómo me llamo? -> No tengo esa información. ¿Cómo te llamas?

CON memoria, hilo 'ana' y hilo 'luis'


  hilo ana  -> ¿Cómo me llamo?: Te llamas Laura.


  hilo luis -> ¿Cómo me llamo?: Te llamas Pedro.


Cada instantánea guardada es consultable: `get_state(config)` devuelve el estado actual (`values`) y los **nodos pendientes** (`next`), y `get_state_history(config)` recorre todas las instantáneas, de la más reciente a la más antigua. Cada `invoke` deja varias instantáneas: una al **recibir la entrada** (con `next` apuntando al nodo de inicio), una **después de cada nodo** y una final. Es la base de la **depuración con viaje en el tiempo**: se puede inspeccionar cualquier punto y **reanudar desde él** pasando su configuración a `invoke(None, configuracion_antigua)`.

In [30]:
config_ana = {"configurable": {"thread_id": "ana"}}
actual = con_memoria.get_state(config_ana)
print("estado actual: mensajes =", len(actual.values["messages"]), "| nodos pendientes =", actual.next)

print("\nhistorial de instantáneas (de la más reciente a la más antigua):")
for instantanea in con_memoria.get_state_history(config_ana):
    print(f"  mensajes = {len(instantanea.values.get('messages', []))} | siguiente = {instantanea.next or 'fin'}")

estado actual: mensajes = 4 | nodos pendientes = ()

historial de instantáneas (de la más reciente a la más antigua):
  mensajes = 4 | siguiente = fin
  mensajes = 3 | siguiente = ('conversar',)
  mensajes = 2 | siguiente = ('__start__',)
  mensajes = 2 | siguiente = fin
  mensajes = 1 | siguiente = ('conversar',)
  mensajes = 0 | siguiente = ('__start__',)


**Intervención humana (*human-in-the-loop*).** A veces el grafo debe **detenerse** y esperar a una persona: aprobar una acción sensible, corregir un borrador, aportar un dato. LangGraph lo resuelve con la función **`interrupt(valor)`**, que **pausa** la ejecución, guarda el estado en el checkpointer y entrega `valor` a quien invocó el grafo (en la clave `__interrupt__` del resultado). Para continuar se invoca de nuevo, con el mismo `thread_id`, pasando **`Command(resume=respuesta)`**: ese `respuesta` es lo que `interrupt(...)` devuelve dentro del nodo.

Un detalle importante: al reanudar, **el nodo que contenía el `interrupt` se vuelve a ejecutar desde su inicio**. Por eso el código con efectos externos (enviar un correo, cobrar) debe ir **después** del `interrupt` o en otro nodo. Construimos un flujo en el que un LLM redacta un correo y una persona lo aprueba, lo edita o lo rechaza antes de "enviarlo":

In [31]:
from langgraph.types import interrupt, Command


class EstadoCorreo(TypedDict):
    peticion: str
    borrador: str
    decision: str
    resultado: str


def redactar_correo(estado: EstadoCorreo):
    return {"borrador": modelo.invoke(f"Redacta un correo breve (3 líneas) para: {estado['peticion']}").text.strip()}


def revisar(estado: EstadoCorreo):
    respuesta = interrupt({"borrador": estado["borrador"], "opciones": ["aprobar", "editar", "rechazar"]})  # pausa aquí
    if respuesta["accion"] == "editar":
        return {"borrador": respuesta["texto"], "decision": "aprobado con cambios"}
    return {"decision": "aprobado" if respuesta["accion"] == "aprobar" else "rechazado"}


def enviar(estado: EstadoCorreo):
    if estado["decision"].startswith("aprobado"):  # el efecto externo va DESPUÉS de la aprobación
        return {"resultado": f"[SIMULACIÓN] Correo enviado:\n{estado['borrador']}"}
    return {"resultado": "No se envió nada."}


flujo_correo = StateGraph(EstadoCorreo)
flujo_correo.add_node("redactar", redactar_correo)
flujo_correo.add_node("revisar", revisar)
flujo_correo.add_node("enviar", enviar)
flujo_correo.add_edge(START, "redactar")
flujo_correo.add_edge("redactar", "revisar")
flujo_correo.add_edge("revisar", "enviar")
flujo_correo.add_edge("enviar", END)
app_correo = flujo_correo.compile(checkpointer=InMemorySaver())

# Caso 1: el flujo se pausa y una persona edita el borrador
cfg = {"configurable": {"thread_id": "correo-1"}}
r = app_correo.invoke({"peticion": "pedir al profesor una prórroga de dos días para el taller 3"}, cfg)
print("El grafo se pausó. Borrador propuesto:\n", r["__interrupt__"][0].value["borrador"])
r = app_correo.invoke(Command(resume={"accion": "editar", "texto": "Estimado profesor: ¿podría darme dos días más para el taller 3? Gracias."}), cfg)
print("\nTras la edición:", r["decision"], "\n", r["resultado"])

# Caso 2: otra persona rechaza el correo
cfg = {"configurable": {"thread_id": "correo-2"}}
app_correo.invoke({"peticion": "pedir al profesor una prórroga de dos días para el taller 3"}, cfg)
print("\nCaso rechazado:", app_correo.invoke(Command(resume={"accion": "rechazar"}), cfg)["resultado"])

El grafo se pausó. Borrador propuesto:
 Asunto: Solicitud de prórroga para entrega del taller 3

Estimado profesor,  
¿Sería posible obtener una prórroga de dos días para entregar el taller 3? Agradezco mucho su comprensión.  

Saludos cordiales,  
[Tu nombre]

Tras la edición: aprobado con cambios 
 [SIMULACIÓN] Correo enviado:
Estimado profesor: ¿podría darme dos días más para el taller 3? Gracias.



Caso rechazado: No se envió nada.


**Streaming de tokens.** Con `stream_mode="messages"`, el grafo emite los **trozos de texto de los LLM** a medida que se generan, junto con metadatos (por ejemplo, el nodo que los produjo), aunque la llamada al modelo ocurra dentro de un nodo. Es lo que permite mostrar la respuesta escribiéndose en una interfaz. Con el enrutador de arriba. Observa que **cada nodo que llama a un modelo** aparece por separado: el clasificador emite el JSON de su salida estructurada y el nodo de respuesta, el texto:

In [32]:
nodos_con_texto = {}
for trozo, metadatos in app_ruta.stream({"pregunta": "¿Qué es una función de pérdida?"}, stream_mode="messages"):
    nodos_con_texto.setdefault(metadatos["langgraph_node"], []).append(trozo.text)
for nodo, partes in nodos_con_texto.items():
    print(f"{nodo:<20} {len(partes):>3} trozos | {''.join(partes)[:110]!r}")

clasificar            12 trozos | '{"categoria":"tecnica"}'
responder_tecnica     52 trozos | 'Una función de pérdida es una medida matemática que cuantifica el error entre las predicciones del modelo y lo'


<a id="agentes-con-langgraph"></a>
<h2 style="color: #007ACC;">Agentes con LangGraph</h2>

Un **agente** es un LLM que **decide por sí mismo** qué hacer en un bucle: razona sobre la tarea, **actúa** llamando a una herramienta, **observa** el resultado y repite hasta poder responder. Es el patrón **ReAct** (*Reasoning + Acting*; Yao et al., 2023), y es exactamente el bucle que escribimos a mano en la sección de *tool calling*:

```text
            ┌──────────────────────────────────────────────┐
            ▼                                              │
 START ──► modelo ──(¿pidió herramientas?)── sí ──► herramientas
            │
            no
            ▼
           END   (respuesta final)
```

En LangGraph ese bucle es un grafo de **dos nodos** y una arista condicional:

- el nodo **`modelo`** invoca al LLM con las herramientas enlazadas y el historial de mensajes;
- el nodo **`herramientas`** (`ToolNode`) ejecuta las `tool_calls` del último mensaje y agrega los `ToolMessage` al estado;
- la arista condicional **`tools_condition`** manda al nodo de herramientas si el último mensaje trae `tool_calls`, o a `END` si no.

Lo construimos **desde cero** con `StateGraph`, para ver que no hay magia:

In [33]:
from langgraph.prebuilt import ToolNode, tools_condition


def construir_agente(m, herramientas, sistema):
    """Agente ReAct desde cero: un nodo de modelo, un nodo de herramientas y una arista condicional."""
    modelo_h = m.bind_tools(herramientas)

    def llamar_modelo(estado: MessagesState):
        return {"messages": [modelo_h.invoke([SystemMessage(sistema)] + estado["messages"])]}

    grafo = StateGraph(MessagesState)
    grafo.add_node("modelo", llamar_modelo)
    grafo.add_node("herramientas", ToolNode(herramientas))
    grafo.add_edge(START, "modelo")
    grafo.add_conditional_edges("modelo", tools_condition, {"tools": "herramientas", END: END})  # ¿pidió herramientas?
    grafo.add_edge("herramientas", "modelo")                                                    # el ciclo
    return grafo.compile()


SISTEMA = "Eres un asistente. Usa las herramientas para datos y cálculos; no inventes cifras. Responde en español de forma breve."
agente_manual = construir_agente(modelo, HERRAMIENTAS, SISTEMA)
agente_manual.get_graph().print_ascii()

r = agente_manual.invoke({"messages": [HumanMessage("Convierte 250 USD a pesos y a ese resultado réstale 100000 usando la calculadora.")]})
print()
mostrar(r["messages"])

             +-----------+             
             | __start__ |             
             +-----------+             
                    *                  
                    *                  
                    *                  
               +--------+              
               | modelo |              
               +--------+              
             ***         ...           
            *               .          
          **                 ..        
+--------------+          +---------+  
| herramientas |          | __end__ |  
+--------------+          +---------+  



usuario      | Convierte 250 USD a pesos y a ese resultado réstale 100000 usando la calculadora.
modelo       | pide: convertir_moneda({'monto': 250, 'origen': 'USD', 'destino': 'COP'}); calculadora({'expresion': '-100000'})
herramienta  | convertir_moneda -> 1025000.0
herramienta  | calculadora -> -100000
modelo       | pide: calculadora({'expresion': '1025000-100000'})
herramienta  | calculadora -> 925000
modelo       | 250 USD convertidos a pesos son 1,025,000 COP. Si a ese resultado le restamos 100,000 COP, el resultado es 925,000 COP.


Lee el recorrido con atención. Aunque la resta **depende** del resultado de la conversión, el modelo pidió **las dos herramientas en la misma ronda**: la calculadora recibió una expresión (`-100000`) escrita sin conocer todavía el resultado de la conversión, y devolvió un valor inservible. Solo en la segunda ronda, con la conversión a la vista, pidió la expresión correcta (`1025000-100000`) y llegó a 925000. En tres de las cuatro ejecuciones que hicimos al preparar el cuaderno ocurrió algo parecido; en una, el modelo pidió las herramientas en orden.

Dos lecciones. El **bucle permite corregirse**: el agente observó el resultado y repitió. Pero el recorrido **no es determinista** y puede incluir llamadas desperdiciadas (más tokens y más latencia). Si quieres evitarlo, puedes indicarlo en el prompt de sistema (*"espera el resultado de una herramienta antes de usarlo en otra"*).

Fíjate en el orden de las llamadas: la resta **depende** del resultado de la conversión. En un workflow fijo, tú decidirías ese orden al escribir el grafo; aquí lo decide el modelo en cada ejecución.

**`create_agent`: el agente ya armado.** Como este patrón es tan común, LangChain lo trae listo: `create_agent(modelo, herramientas, system_prompt=...)` construye el mismo grafo y le añade opciones: salida estructurada, memoria, *middleware* para controlar el comportamiento, contexto de ejecución y más. Es la forma recomendada hoy (antes se usaba `langgraph.prebuilt.create_react_agent`).

In [34]:
from langchain.agents import create_agent

agente = create_agent(modelo, tools=HERRAMIENTAS, system_prompt=SISTEMA)
print("Nodos de create_agent:", list(agente.get_graph().nodes), "| nodos del agente manual:", list(agente_manual.get_graph().nodes))

r = agente.invoke({"messages": [HumanMessage("¿Qué ciudad es más cálida hoy, Cali o Bogotá? Dime las temperaturas.")]})
mostrar(r["messages"])

Nodos de create_agent: ['__start__', 'model', 'tools', '__end__'] | nodos del agente manual: ['__start__', 'modelo', 'herramientas', '__end__']


usuario      | ¿Qué ciudad es más cálida hoy, Cali o Bogotá? Dime las temperaturas.
modelo       | pide: clima({'ciudad': 'Cali'}); clima({'ciudad': 'Bogotá'})
herramienta  | clima -> En Cali: 28 °C, soleado.
herramienta  | clima -> En Bogotá: 16 °C, nublado.
modelo       | Hoy Cali es más cálida con 28 °C y soleado, mientras que en Bogotá hay 16 °C y está nublado.


El resultado es equivalente, con menos código. ¿Y funciona igual de bien con los tres proveedores? Los comparamos con **6 preguntas de verificación automática**: la respuesta es correcta si el texto final contiene todos los datos esperados (números extraídos del texto, para tolerar formatos como `1.025.000` o `1,025,000`). Si una herramienta lanza una excepción, el agente se detiene y lo contamos como fallo (más abajo vemos por qué). Medimos el éxito, las llamadas a herramientas, las llamadas al modelo, los **tokens totales** (sumando el `usage_metadata` que trae cada respuesta del modelo) y el tiempo. Una de las preguntas necesita **buscar en los documentos**, y otra encadena dos pasos dependientes.

In [35]:
PREGUNTAS = [
    ("Dime el clima de Tunja y cuántos pesos son 250 USD.", [1025000, 14]),
    ("¿Cuánto es ((125 * 8) + 17) / 3?", [339]),
    ("¿Cuántos euros son 90000 pesos colombianos?", [20]),
    ("Convierte 250 USD a pesos y a ese resultado réstale 100000 usando la calculadora.", [925000]),
    ("¿Qué ciudad es más cálida hoy, Cali o Bogotá? Dime las temperaturas.", ["cali", 28, 16]),
    ("¿Cuántas personas hay por grupo en el proyecto del módulo 4 del curso?", [2, 3]),
]


def numeros_en(texto):
    """Números del texto, tolerando separadores de miles (1.025.000 o 1,025,000) y decimales."""
    hallados = []
    for crudo in re.findall(r"\d[\d.,]*", texto):
        s = crudo.rstrip(".,")
        s = re.sub(r"[.,]", "", s) if re.fullmatch(r"\d{1,3}([.,]\d{3})+", s) else s.replace(",", ".")
        try:
            hallados.append(float(s))
        except ValueError:
            pass
    return hallados


def cumple(texto, esperados):
    numeros = numeros_en(texto)
    return all((e in numeros) if isinstance(e, (int, float)) else (e in texto.lower()) for e in esperados)


def evaluar_agente(ag, pregunta, esperados):
    inicio = time.perf_counter()
    try:
        mensajes = ag.invoke({"messages": [HumanMessage(pregunta)]})["messages"]
    except Exception as error:   # una excepción de una herramienta detiene al agente: la contamos como fallo
        return {"correcta": False, "error": type(error).__name__, "herramientas": 0, "llamadas al modelo": 0, "tokens": 0,
                "segundos": time.perf_counter() - inicio}
    respuestas_del_modelo = [m for m in mensajes if isinstance(m, AIMessage)]
    return {"correcta": cumple(mensajes[-1].text, esperados), "error": "",
            "herramientas": sum(isinstance(m, ToolMessage) for m in mensajes),
            "llamadas al modelo": len(respuestas_del_modelo),
            "tokens": sum(m.usage_metadata["total_tokens"] for m in respuestas_del_modelo),  # cada respuesta trae su consumo
            "segundos": time.perf_counter() - inicio}


agentes = {p: create_agent(m, tools=HERRAMIENTAS, system_prompt=SISTEMA) for p, m in MODELOS.items()}
detalle = []
for proveedor, ag in agentes.items():
    with ThreadPoolExecutor(3) as pool:
        resultados = list(pool.map(lambda c: evaluar_agente(ag, *c), PREGUNTAS))
    detalle += [{"proveedor": proveedor, "pregunta": p[:60], **r} for r, (p, _) in zip(resultados, PREGUNTAS)]
detalle = pd.DataFrame(detalle)
display(detalle.groupby("proveedor").agg(**{"respuestas correctas (de 6)": ("correcta", "sum"), "herramientas por pregunta": ("herramientas", "mean"),
                                           "llamadas al modelo": ("llamadas al modelo", "mean"), "tokens por pregunta": ("tokens", "mean"),
                                           "segundos por pregunta": ("segundos", "mean")}).round(1))
print("Preguntas respondidas incorrectamente (y, si el agente se detuvo, la excepción):")
display(detalle[~detalle.correcta][["proveedor", "pregunta", "error"]].set_index("proveedor"))

,respuestas correctas (de 6),herramientas por pregunta,llamadas al modelo,tokens por pregunta,segundos por pregunta
proveedor,,,,,
Google,6,1.5,2.2,961.5,1.9
Ollama,2,1.2,1.7,590.0,0.8
OpenAI,6,1.7,2.2,736.5,1.8


Preguntas respondidas incorrectamente (y, si el agente se detuvo, la excepción):


,pregunta,error
proveedor,,
Ollama,¿Cuánto es ((125 * 8) + 17) / 3?,
Ollama,Convierte 250 USD a pesos y a ese resultado réstale 100000 u,
Ollama,"¿Qué ciudad es más cálida hoy, Cali o Bogotá? Dime las tempe",SyntaxError
Ollama,¿Cuántas personas hay por grupo en el proyecto del módulo 4,


- **OpenAI y Google resolvieron las 6 preguntas**, incluidas la encadenada y la de documentos, con 1,5 a 1,7 herramientas y unas 2,2 llamadas al modelo por pregunta. Google gastó más tokens por pregunta (unos 960 frente a unos 740 de OpenAI).
- **El modelo local pequeño (`llama3.2:3b`) acertó 2 de 6.** Fue el más rápido y el más barato (0,8 s y unos 590 tokens por pregunta), pero porque se equivoca antes. En las ejecuciones que hicimos sus fallos fueron de tres tipos: **expresiones mal escritas** (pasó `(125 * 8) + 17 / 3`, sin los paréntesis, y obtuvo 1005,67 en lugar de 339), **herramienta equivocada** (la calculadora en lugar de buscar en los documentos) y **datos inventados** (una tasa de cambio que no existe). En una pregunta escribió como "expresión" algo que no era aritmética, la calculadora lanzó `SyntaxError` y el agente **se detuvo** (columna `error`): justo el caso que resolvemos con `ToolErrorMiddleware` más abajo.
- Es una advertencia sobre el **tamaño del modelo**: tool calling y razonamiento en varios pasos exigen más de lo que da un modelo de 3 000 millones de parámetros. Prueba un modelo local mayor (el último ejercicio) antes de concluir que "los modelos locales no sirven para agentes".

**Memoria de conversación.** Igual que en cualquier grafo, se añade con un *checkpointer* y se usa con un `thread_id`. El agente recuerda lo hablado y puede resolver referencias como *"¿y en Cali?"*:

In [36]:
agente_con_memoria = create_agent(modelo, tools=HERRAMIENTAS, system_prompt=SISTEMA, checkpointer=InMemorySaver())
hilo = {"configurable": {"thread_id": "conversacion-1"}}
for turno in ["¿Qué clima hace en Tunja?", "¿Y en Cali?", "¿Cuál de las dos ciudades es más cálida?"]:
    r = agente_con_memoria.invoke({"messages": [HumanMessage(turno)]}, hilo)
    print(f"usuario: {turno}\nagente : {r['messages'][-1].text.strip()}\n")
print("mensajes guardados en el hilo:", len(agente_con_memoria.get_state(hilo).values["messages"]))

usuario: ¿Qué clima hace en Tunja?
agente : En Tunja hace 14 °C y hay llovizna.



usuario: ¿Y en Cali?
agente : En Cali hace 28 °C y está soleado.



usuario: ¿Cuál de las dos ciudades es más cálida?
agente : Cali es más cálida que Tunja.

mensajes guardados en el hilo: 10


**Memoria de corto y largo plazo.** El `checkpointer` da memoria **dentro de un hilo** (corto plazo). Para recordar cosas **entre conversaciones** (preferencias de un usuario, hechos aprendidos), `create_agent` acepta además un `store`, un almacén clave-valor compartido entre hilos. Y como el historial crece con cada turno, el *middleware* `SummarizationMiddleware` puede resumir los mensajes antiguos cuando el contexto se hace grande (lo vemos en la sección siguiente).

**Streaming del agente.** `stream` permite observar al agente trabajando. Con `stream_mode="updates"` vemos cada paso del ciclo; con `"messages"`, el texto final llega token por token:

In [37]:
print("Pasos del agente (stream_mode='updates'):")
for evento in agente.stream({"messages": [HumanMessage("Dime el clima de Cali y convierte 100 EUR a pesos.")]}, stream_mode="updates"):
    for nodo, actualizacion in evento.items():
        ultimo = actualizacion["messages"][-1]
        detalle_paso = ("pide " + ", ".join(c["name"] for c in ultimo.tool_calls)) if getattr(ultimo, "tool_calls", None) else ultimo.text.strip()[:90]
        print(f"  {nodo:<7} -> {type(ultimo).__name__}: {detalle_paso}")

print("\nTexto final, token por token (stream_mode='messages'):")
texto = ""
for trozo, metadatos in agente.stream({"messages": [HumanMessage("¿Cuánto es 15 % de 2400?")]}, stream_mode="messages"):
    if metadatos["langgraph_node"] == "model" and trozo.text:
        texto += trozo.text
print(" ", texto.strip())

Pasos del agente (stream_mode='updates'):


  model   -> AIMessage: pide clima, convertir_moneda
  tools   -> ToolMessage: En Cali: 28 °C, soleado.
  tools   -> ToolMessage: 450000.0


  model   -> AIMessage: En Cali está soleado con una temperatura de 28 °C. Además, 100 EUR equivalen a 450,000 pes

Texto final, token por token (stream_mode='messages'):


  El 15 % de 2400 es 360.


**Respuesta estructurada.** Cuando la salida del agente alimenta a otro programa, `response_format` le indica un esquema de Pydantic: al terminar, el agente produce un objeto validado en `resultado["structured_response"]`, además de los mensajes:

In [38]:
class ReporteClima(BaseModel):
    ciudad: str
    temperatura_c: int = Field(description="Temperatura en grados Celsius")
    llueve: bool


agente_estructurado = create_agent(modelo, tools=HERRAMIENTAS, system_prompt=SISTEMA, response_format=ReporteClima)
r = agente_estructurado.invoke({"messages": [HumanMessage("Reporte del clima de Tunja")]})
print(r["structured_response"])

ciudad='Tunja' temperatura_c=14 llueve=True


**Contexto de ejecución.** A veces una herramienta necesita datos que **no deben pasar por el modelo**: el usuario autenticado, su organización, una conexión a una base de datos. `create_agent` acepta un `context_schema` (una clase de datos) y los valores concretos se entregan en cada `invoke(..., context=...)`. Las herramientas los reciben mediante un parámetro `runtime: ToolRuntime`, que el modelo **no ve** (no aparece en el esquema). Así el modelo no puede pedirle a la herramienta el saldo de *otro* usuario:

In [39]:
from dataclasses import dataclass
from langchain.tools import ToolRuntime

SALDOS = {"laura": 120_000, "pedro": 45_000}


@dataclass
class Contexto:
    usuario: str


@tool
def mi_saldo(runtime: ToolRuntime[Contexto]) -> str:
    """Consulta el saldo del usuario que está conversando con el asistente."""
    usuario = runtime.context.usuario   # viene del contexto, no del modelo
    return f"Saldo de {usuario}: ${SALDOS[usuario]:,}".replace(",", ".")


agente_contexto = create_agent(modelo, tools=[mi_saldo], context_schema=Contexto, system_prompt="Eres un asistente bancario breve.")
for usuario in ["laura", "pedro"]:
    r = agente_contexto.invoke({"messages": [HumanMessage("¿Cuál es mi saldo?")]}, context=Contexto(usuario=usuario))
    print(f"{usuario:<6} -> {r['messages'][-1].text.strip()}")
print("\nEsquema que ve el modelo para mi_saldo (sin 'runtime'):", convert_to_openai_tool(mi_saldo)["function"]["parameters"])

laura  -> Tu saldo actual es de $120.000. ¿Hay algo más en lo que pueda ayudarte?


pedro  -> Tu saldo actual es de $45.000. ¿Hay algo más en lo que pueda ayudarte?

Esquema que ve el modelo para mi_saldo (sin 'runtime'): {'properties': {}, 'type': 'object'}


<a id="control-y-seguridad-de-agentes"></a>
<h2 style="color: #007ACC;">Control y seguridad de agentes</h2>

Un agente **decide por su cuenta** cuántas veces llama al modelo y qué herramientas usa. Eso lo hace potente y, a la vez, riesgoso: puede **entrar en un ciclo** (costo ilimitado), **equivocarse con una herramienta** (efectos reales), o ser **manipulado** por un texto malicioso que lea (inyección de prompts, cuaderno 18, ahora con herramientas que *actúan*). Los controles se organizan en capas:

| Capa | Mecanismo | Para qué |
|---|---|---|
| **Límite del grafo** | `recursion_limit` en la configuración | Corta el ciclo tras N pasos con `GraphRecursionError` |
| **Límites de uso** | `ModelCallLimitMiddleware`, `ToolCallLimitMiddleware` | Topes de llamadas por ejecución o por hilo |
| **Errores de herramientas** | `ToolErrorMiddleware`, `ToolRetryMiddleware` | Devolver el error al modelo o reintentar |
| **Aprobación humana** | `HumanInTheLoopMiddleware` | Pausar antes de herramientas sensibles |
| **Respaldo y reintentos del modelo** | `ModelFallbackMiddleware`, `ModelRetryMiddleware` | Cambiar de modelo o reintentar ante fallos |
| **Contexto y datos** | `SummarizationMiddleware`, `PIIMiddleware` | Resumir el historial largo; detectar o enmascarar datos personales |
| **Planificación** | `TodoListMiddleware` | Una lista de tareas para que el agente organice trabajos largos |
| **Selección de herramientas** | `LLMToolSelectorMiddleware` | Preseleccionar las herramientas relevantes cuando hay muchas |

Un ***middleware*** es un componente que se **engancha al ciclo del agente** (antes o después de cada llamada al modelo o a una herramienta) para observar, modificar o detener la ejecución. Se pasan en la lista `middleware=[...]` de `create_agent` y se pueden combinar. Estos son los que ofrece la versión instalada:

In [40]:
import langchain.agents.middleware as mw

print([nombre for nombre in dir(mw) if nombre.endswith("Middleware") and nombre != "AgentMiddleware"])

['ContextEditingMiddleware', 'FilesystemFileSearchMiddleware', 'HumanInTheLoopMiddleware', 'LLMToolSelectorMiddleware', 'ModelCallLimitMiddleware', 'ModelFallbackMiddleware', 'ModelRetryMiddleware', 'PIIMiddleware', 'ProviderToolSearchMiddleware', 'ShellToolMiddleware', 'SummarizationMiddleware', 'TodoListMiddleware', 'ToolCallLimitMiddleware', 'ToolErrorMiddleware', 'ToolRetryMiddleware']


**1. Un ciclo sin fin.** Creamos una herramienta que siempre pide reintentar y un agente al que le decimos que no se rinda. Sin límites, el agente seguiría indefinidamente (y gastando). Lo detenemos de dos formas: con `recursion_limit` y con `ToolCallLimitMiddleware`, que en lugar de lanzar una excepción **termina con gracia** y avisa:

In [41]:
from langgraph.errors import GraphRecursionError
from langchain.agents.middleware import ToolCallLimitMiddleware


@tool
def siempre_falla(dato: str) -> str:
    """Intenta procesar un dato."""
    return "Error temporal: vuelve a intentarlo."


instruccion = "Debes llamar a siempre_falla hasta que funcione; nunca te rindas."
pedido = {"messages": [HumanMessage("Procesa el dato 'a' con siempre_falla.")]}

# a) recursion_limit: el grafo lanza una excepción al llegar al tope
try:
    create_agent(modelo, tools=[siempre_falla], system_prompt=instruccion).invoke(pedido, {"recursion_limit": 6})
except GraphRecursionError as error:
    print("recursion_limit ->", type(error).__name__)

# b) ToolCallLimitMiddleware: termina con gracia tras N llamadas a herramientas
con_limite = create_agent(modelo, tools=[siempre_falla], system_prompt=instruccion,
                          middleware=[ToolCallLimitMiddleware(run_limit=2, exit_behavior="end")])
r = con_limite.invoke(pedido)
print("ToolCallLimitMiddleware ->", [type(m).__name__ for m in r["messages"]], "\n  último mensaje:", r["messages"][-1].text[:90])

recursion_limit -> GraphRecursionError


ToolCallLimitMiddleware -> ['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage', 'ToolMessage', 'AIMessage', 'ToolMessage', 'AIMessage'] 
  último mensaje: Tool call limit reached: run limit exceeded (3/2 calls).


Con `run_limit=2` el agente hizo dos llamadas a herramientas y, al intentar la tercera, el *middleware* terminó la ejecución con un mensaje (`3/2 calls`) en lugar de lanzar una excepción. Con `recursion_limit`, en cambio, se lanza `GraphRecursionError`, que tu código debe capturar. Usa los dos: el primero para un final ordenado, el segundo como red de seguridad.

**2. Errores de las herramientas.** Los errores de **validación de argumentos** (un valor fuera de la lista permitida, un tipo equivocado) los convierte `ToolNode` en un `ToolMessage` con `status="error"` **antes** de ejecutar la herramienta, y así el modelo los ve y puede corregirse. Pero las **excepciones de tu propio código** (la cuenta que no existe) **detienen el agente por defecto**, para no mostrar detalles internos al modelo ni al usuario sin querer. Si quieres que el modelo las reciba y se recupere, lo declaras con `ToolErrorMiddleware(on_error=...)`: la función recibe la excepción y la petición, y devuelve el **texto** que verá el modelo (o `None` para dejar pasar la excepción). La documentación recomienda devolver el **tipo** del error y no el mensaje crudo, que puede contener información sensible:

In [42]:
from langchain.agents.middleware import ToolErrorMiddleware

pregunta_saldo = {"messages": [HumanMessage("¿Cuál es el saldo de la cuenta 999?")]}
sistema_banco = "Eres un asistente bancario breve."

try:
    create_agent(modelo, tools=[consultar_saldo], system_prompt=sistema_banco).invoke(pregunta_saldo)
except Exception as error:
    print(f"Sin middleware: el agente se detuvo con {type(error).__name__}: {error}")


def al_fallar(error: Exception, peticion):
    if isinstance(error, ValueError):  # solo manejamos los errores que anticipamos
        return f"La herramienta {peticion.tool_call['name']} falló ({type(error).__name__}). Revisa los datos e informa al usuario."
    return None                         # cualquier otro error se propaga


tolerante = create_agent(modelo, tools=[consultar_saldo], system_prompt=sistema_banco, middleware=[ToolErrorMiddleware(al_fallar)])
print("\nCon ToolErrorMiddleware:")
mostrar(tolerante.invoke(pregunta_saldo)["messages"])

Sin middleware: el agente se detuvo con ValueError: La cuenta 999 no existe

Con ToolErrorMiddleware:


usuario      | ¿Cuál es el saldo de la cuenta 999?
modelo       | pide: consultar_saldo({'cuenta': '999'})
herramienta  | consultar_saldo -> La herramienta consultar_saldo falló (ValueError). Revisa los datos e informa al usuario.  [ERROR]
modelo       | El número de cuenta 999 no es válido. Por favor, verifica el número de cuenta e inténtalo de nuevo.


**3. Aprobación humana de herramientas sensibles.** `HumanInTheLoopMiddleware` pausa el agente **antes de ejecutar** las herramientas que indiques, y espera una decisión por cada llamada: **`approve`** (ejecutar tal cual), **`edit`** (ejecutar con argumentos corregidos) o **`reject`** (no ejecutar, con un mensaje para el modelo). Funciona con el mismo mecanismo de `interrupt` y `Command(resume=...)` de LangGraph, así que **exige un *checkpointer***. Las herramientas inofensivas (`False`) se ejecutan sin pausa. Probamos las tres decisiones con una herramienta que "envía correos":

In [43]:
from langchain.agents.middleware import HumanInTheLoopMiddleware


@tool
def enviar_correo(destinatario: str, asunto: str) -> str:
    """Envía un correo electrónico (acción con efectos externos)."""
    return f"[SIMULACIÓN] Correo enviado a {destinatario} con asunto «{asunto}»"


def agente_con_aprobacion():
    return create_agent(modelo, tools=[clima, enviar_correo], checkpointer=InMemorySaver(),
                        middleware=[HumanInTheLoopMiddleware(interrupt_on={"enviar_correo": True, "clima": False})])


PEDIDO_CORREO = {"messages": [HumanMessage("Envía un correo a ana@ejemplo.co con asunto Hola Ana")]}
DECISIONES = {
    "approve": {"type": "approve"},
    "edit": {"type": "edit", "edited_action": {"name": "enviar_correo", "args": {"destinatario": "luis@ejemplo.co", "asunto": "Hola Luis"}}},
    "reject": {"type": "reject", "message": "No envíes correos sin que yo revise el contenido."},
}
for nombre, decision in DECISIONES.items():
    ag = agente_con_aprobacion()
    hilo = {"configurable": {"thread_id": nombre}}
    pausa = ag.invoke(PEDIDO_CORREO, hilo)["__interrupt__"][0].value["action_requests"][0]
    r = ag.invoke(Command(resume={"decisions": [decision]}), hilo)
    herramienta = next((m for m in r["messages"] if isinstance(m, ToolMessage)), None)
    print(f"[{nombre}] el agente quería: {pausa['args']}")
    print(f"          herramienta: {herramienta.status} | {herramienta.content[:260]!r}")
    print(f"          modelo dice: {r['messages'][-1].text.strip()[:110]}\n")

[approve] el agente quería: {'destinatario': 'ana@ejemplo.co', 'asunto': 'Hola Ana'}
          herramienta: success | '[SIMULACIÓN] Correo enviado a ana@ejemplo.co con asunto «Hola Ana»'
          modelo dice: El correo con asunto "Hola Ana" ha sido enviado a ana@ejemplo.co. ¿Hay algo más en lo que pueda ayudarte?



[edit] el agente quería: {'destinatario': 'ana@ejemplo.co', 'asunto': 'Hola Ana'}
          herramienta: success | 'Note: a human reviewer replaced this tool call before it ran. The call recorded in your message is the one you produced, not the one that executed. This was intentional and authorized. Do not re-issue your original call. Executed instead: enviar_correo with ar'
          modelo dice: El correo con asunto "Hola Ana" ha sido enviado a ana@ejemplo.co. ¿Hay algo más en lo que pueda ayudarte?



[reject] el agente quería: {'destinatario': 'ana@ejemplo.co', 'asunto': 'Hola Ana'}
          herramienta: error | 'User rejected the tool call for `enviar_correo` with reason: No envíes correos sin que yo revise el contenido.'
          modelo dice: Antes de enviar el correo, ¿podrías indicarme qué contenido quieres que incluya en el mensaje? Así puedo prepa



Lee con atención el caso `edit`: la persona cambió los argumentos y la herramienta se ejecutó **con los argumentos editados** (el `ToolMessage` es una nota que explica la sustitución y qué se ejecutó), pero el modelo, en su mensaje final, sigue hablando de la llamada **original** (`ana@ejemplo.co`). Cuando editas una acción, **no des por hecho que el texto del modelo describe lo que realmente se ejecutó**: la fuente de verdad es el `ToolMessage` y el registro de la herramienta.

**Principios de seguridad para agentes** (resumen de lo anterior y del cuaderno 18):

1. **Mínimo privilegio**: solo las herramientas indispensables, con los permisos justos.
2. **Aprobación humana** para toda acción irreversible o con efectos externos (pagos, correos, borrados, escrituras).
3. **Límites duros** de pasos, llamadas y tiempo, y presupuesto de tokens por ejecución.
4. **Tratar como no confiable** todo texto que el agente lea (páginas web, correos, documentos recuperados): puede contener instrucciones inyectadas. Un agente con herramientas que *actúan* y texto externo es la combinación más peligrosa.
5. **Validar** argumentos y salidas en código, no solo en el prompt.
6. **Registrar** cada paso (trazas) para poder auditar qué hizo y por qué.

<a id="sistemas-multiagente"></a>
<h2 style="color: #007ACC;">Sistemas multiagente</h2>

Cuando un agente acumula demasiadas herramientas o instrucciones, su rendimiento se degrada: el contexto se llena y la elección de herramientas empeora. Una solución es **dividir el trabajo** entre varios agentes especializados, cada uno con pocas herramientas y un prompt corto. Patrones frecuentes:

| Patrón | Idea | Cuándo |
|---|---|---|
| **Supervisor** | Un agente coordinador recibe la petición y **delega** en especialistas, que le devuelven resultados | Tareas con subtareas claramente distintas |
| **Subagentes como herramientas** | Cada especialista se expone al coordinador como una **herramienta** | La forma más simple de implementar el supervisor |
| **Traspaso (*handoff*)** | Un agente **cede el control** a otro y la conversación continúa con el nuevo | Flujos por etapas (ventas → soporte) |
| **Enrutador** | Una clasificación inicial envía la petición al agente adecuado (como el enrutador que hicimos) | Dominios separados |

Implementamos el patrón más simple: dos **subagentes** (un matemático y un redactor), cada uno un agente completo, expuestos a un **coordinador** como herramientas. El coordinador no sabe cómo trabajan por dentro: solo ve sus descripciones.

In [44]:
@tool
def consultar_matematico(problema: str) -> str:
    """Delega un problema matemático a un especialista. Devuelve solo el resultado numérico."""
    especialista = create_agent(modelo, tools=[calculadora], system_prompt="Eres matemático. Usa la calculadora y responde solo con el resultado.")
    return especialista.invoke({"messages": [HumanMessage(problema)]})["messages"][-1].text


@tool
def consultar_redactor(texto: str) -> str:
    """Delega la redacción de un resumen de una línea a un especialista."""
    especialista = create_agent(modelo, tools=[], system_prompt="Eres redactor. Resume el texto en una sola línea clara.")
    return especialista.invoke({"messages": [HumanMessage(texto)]})["messages"][-1].text


coordinador = create_agent(modelo, tools=[consultar_matematico, consultar_redactor],
                           system_prompt="Eres un coordinador. Delega los cálculos en el matemático y la redacción en el redactor; "
                                         "espera el resultado del cálculo antes de pedir el resumen.")
r = coordinador.invoke({"messages": [HumanMessage("Calcula el 15 % de 2400 y luego redacta una frase que presente ese resultado.")]})
mostrar(r["messages"])

usuario      | Calcula el 15 % de 2400 y luego redacta una frase que presente ese resultado.
modelo       | pide: consultar_matematico({'problema': 'Calcular el 15% de 2400'})
herramienta  | consultar_matematico -> El 15% de 2400 es 360.
modelo       | pide: consultar_redactor({'texto': 'El 15% de 2400 es 360.'})
herramienta  | consultar_redactor -> El 15% de 2400 equivale a 360.
modelo       | El 15% de 2400 equivale a 360.


**Una advertencia.** Los sistemas multiagente **multiplican el costo y la latencia** (cada subagente hace sus propias llamadas al modelo) y añaden puntos de fallo: la comunicación entre agentes pasa por texto, y los errores se propagan. La regla práctica es la misma de siempre: empieza con **un solo agente bien hecho** y divídelo solo cuando midas que un agente único falla por exceso de herramientas o de contexto.

**RAG agéntico.** Combinemos lo visto: un agente con el recuperador y la calculadora, ante una pregunta que **necesita buscar un dato** y **luego calcular con él**: *"Si cada persona de un grupo del proyecto del módulo 4 pagara 15 000 pesos, ¿cuál sería el máximo que pagaría un grupo?"*. El agente tiene que recuperar el tamaño máximo del grupo (3 personas) y multiplicar:

In [45]:
agente_rag = create_agent(modelo, tools=[buscar_documentos, calculadora], system_prompt=SISTEMA)
r = agente_rag.invoke({"messages": [HumanMessage("Si cada persona de un grupo del proyecto del módulo 4 pagara 15000 pesos, ¿cuál sería el máximo que pagaría un grupo?")]})
mostrar(r["messages"])

usuario      | Si cada persona de un grupo del proyecto del módulo 4 pagara 15000 pesos, ¿cuál sería el máximo que pagaría un grupo?
modelo       | pide: buscar_documentos({'consulta': 'módulo 4 proyecto tamaño grupo'})
herramienta  | buscar_documentos -> El proyecto del módulo 4 se desarrolla en grupos de 2 a 3 personas y debe poder ejecutarse en Google Colab con una GPU T4.

El proyecto del módulo 4 exige una presentación de 10 minutos con los hallazgos principales y una auditoría ética con su tarjeta del modelo.

Módulo 4 (semanas 7 y 8): atención y modelos preentrenados: Transformers, BERT y GPT con fine-tuning, aplicaciones de modelos preentrenados, prompt engineering, LangChain y LangGraph, y ética en PLN.

Cada módulo tiene cuadernos de clase, talleres prácticos con su solución y un proyecto de aplicación.
modelo       | pide: calculadora({'expresion': '3 * 15000'})
herramienta  | calculadora -> 45000
modelo       | El máximo que pagaría un grupo del proyecto del módulo 4, si ca

<a id="observabilidad-y-costos"></a>
<h2 style="color: #007ACC;">Observabilidad y costos</h2>

Un agente es difícil de depurar porque su recorrido **cambia** en cada ejecución. Necesitas poder ver **qué hizo y cuánto costó**. Hay tres niveles:

1. **El resultado**: la lista de mensajes que devuelve `invoke` es la traza más simple (la usamos con `mostrar`).
2. **Callbacks**: funciones que reciben eventos (inicio y fin de cada llamada a un modelo o herramienta). `UsageMetadataCallbackHandler`, por ejemplo, suma los tokens de toda una ejecución (lo usamos abajo).
3. **LangSmith**: servicio de LangChain para **trazas, evaluación y monitoreo**. Se activa solo con variables de entorno (`LANGSMITH_TRACING=true` y `LANGSMITH_API_KEY`), sin cambiar el código, y registra cada paso del grafo con sus entradas, salidas, tokens y latencias. **No lo activamos aquí**: enviaría los datos de las ejecuciones a un servicio externo.

**El costo de un agente crece con cada paso**, porque cada llamada al modelo **reenvía todo el historial**: la pregunta, los mensajes anteriores y los resultados de las herramientas. Lo vemos con los tokens de entrada de cada llamada al modelo en una misma ejecución, frente a una sola llamada sin herramientas:

In [46]:
from langchain_core.callbacks import UsageMetadataCallbackHandler

pregunta_larga = "Convierte 250 USD a pesos, réstale 100000 con la calculadora, y dime también el clima de Tunja."
uso = UsageMetadataCallbackHandler()   # un callback que suma los tokens de todas las llamadas de la ejecución
r = agente.invoke({"messages": [HumanMessage(pregunta_larga)]}, {"callbacks": [uso]})
print("Tokens según el callback:", uso.usage_metadata)
llamadas_al_modelo = [m for m in r["messages"] if isinstance(m, AIMessage)]
tabla = pd.DataFrame({
    "llamada al modelo": range(1, len(llamadas_al_modelo) + 1),
    "pide": [", ".join(c["name"] for c in m.tool_calls) or "(respuesta final)" for m in llamadas_al_modelo],
    "tokens de entrada": [m.usage_metadata["input_tokens"] for m in llamadas_al_modelo],
    "tokens de salida": [m.usage_metadata["output_tokens"] for m in llamadas_al_modelo]}).set_index("llamada al modelo")
display(tabla)

sola = modelo.invoke(pregunta_larga)
print(f"Tokens totales del agente: {int(tabla[['tokens de entrada', 'tokens de salida']].sum().sum())} | una sola llamada sin herramientas: {sola.usage_metadata['total_tokens']} "
      f"(que no puede calcular ni consultar el clima con fiabilidad)")

Tokens según el callback: {'gpt-4.1-mini-2025-04-14': {'input_token_details': {'cache_read': 0, 'audio': 0}, 'total_tokens': 1154, 'input_tokens': 1026, 'output_tokens': 128, 'output_token_details': {'reasoning': 0, 'audio': 0}}}


,pide,tokens de entrada,tokens de salida
llamada al modelo,,,
1,"convertir_moneda, clima",270,59
2,calculadora,362,22
3,(respuesta final),394,47


Tokens totales del agente: 1154 | una sola llamada sin herramientas: 185 (que no puede calcular ni consultar el clima con fiabilidad)


El agente hizo tres llamadas al modelo y los **tokens de entrada crecieron** en cada una (270, 362 y 394), porque cada llamada reenvía el historial completo: la pregunta, las peticiones de herramientas y sus resultados. En total gastó 1154 tokens, unas **6 veces** los 185 de una sola llamada sin herramientas. Es el precio de poder actuar: el costo y la latencia crecen con el número de pasos, y con ellos el costo de un agente atascado en un ciclo, de ahí los límites de la sección anterior.

<a id="cuando-usar-estas-librerias"></a>
<h2 style="color: #007ACC;">Cuándo usar estas librerías</h2>

LangChain y LangGraph no son obligatorios: todo lo que hicimos se puede escribir con el SDK de cada proveedor (como en el cuaderno 18). Su valor depende del tamaño del problema:

| Situación | Recomendación |
|---|---|
| Una o dos llamadas a un modelo, sin herramientas | **El SDK directo.** Una dependencia grande no aporta |
| Cambiar de proveedor o usar varios (respaldo, comparación) | **LangChain**: interfaz común, `with_fallbacks`, `init_chat_model` |
| Prompts con variables, salida estructurada, RAG | **LangChain**: componentes listos y composición con `|` |
| Flujos con ciclos, estado, memoria, pausas humanas | **LangGraph**: es para lo que existe |
| Un agente con herramientas | **`create_agent`**; baja a LangGraph si necesitas control fino del flujo |

**Costos de adoptarlas.** (1) **Abstracción con fugas**: vimos que `parallel_tool_calls` no existe en todos los proveedores y que `.content` cambia de tipo. (2) **Volatilidad**: las APIs cambian de versión en versión (la creación de agentes se movió de paquete; hay métodos que ya se marcan como obsoletos; hasta el modelo de Google que usamos en el cuaderno 18 dejó de estar disponible). Conviene **fijar versiones** en un `requirements.txt`. (3) **Depuración más difícil**: más capas entre tu código y la llamada real; las trazas (LangSmith u otras) pasan a ser casi imprescindibles. (4) **Más superficie de ataque**: un agente con herramientas amplía lo que puede salir mal.

Hay alternativas: otros frameworks de agentes y de RAG, y los SDK oficiales de agentes de los propios proveedores. Los **conceptos** (tool calling, ciclo agente-herramienta, estado, memoria, aprobación humana, límites) son los mismos; lo que cambia es la sintaxis.

**Resumen del cuaderno:**

- **LangChain** da los componentes (modelos de chat con interfaz común, plantillas, salida estructurada, herramientas, RAG) y los compone con `|` (LCEL); **LangGraph** da el motor de ejecución (grafos con estado, ciclos, memoria, pausas humanas, streaming).
- Todo componente es un **Runnable** con `invoke`, `stream`, `batch` y sus versiones asíncronas, lo que permite paralelizar y tolerar fallos (`with_retry`, `with_fallbacks`) casi sin código.
- En el **tool calling**, el modelo no ejecuta nada: **decide** la herramienta y los argumentos, y **tu código** la ejecuta y le devuelve un `ToolMessage`. Repetido en bucle, es un agente.
- La **descripción de la herramienta** es lo único que el modelo ve: con descripciones vagas, la elección correcta cayó de forma marcada, sobre todo en el modelo local pequeño.
- Un **grafo** se define con estado, nodos, aristas (fijas o condicionales) y reductores. Un **workflow** fija el camino en código; un **agente** deja que el LLM lo decida.
- Con un **checkpointer** el grafo recuerda por hilo y se puede inspeccionar y reanudar; con **`interrupt`** se pausa a esperar a una persona.
- `create_agent` arma el agente ReAct y se controla con **middleware** (límites, errores, aprobación humana, resumen). **Un agente sin límites ni aprobación es un riesgo.**
- Un agente **cuesta más** que una llamada: cada paso reenvía el historial. Los sistemas multiagente lo multiplican; hay que medir antes de dividir.

**Para practicar:**

1. Añade una herramienta `buscar_en_internet` simulada y comprueba si el modelo sigue eligiendo bien con 5 herramientas. Repite el experimento de descripciones con ella.
2. Cambia el verificador del ciclo de reflexión por un segundo LLM como crítico y compara el número de iteraciones con el verificador determinista.
3. Haz que el agente con memoria guarde las preferencias del usuario en un `store` y las recuerde en un hilo nuevo.
4. Añade `SummarizationMiddleware` a un agente con conversación larga y mide cuántos tokens de entrada ahorra.
5. Escribe una reseña de ataque (inyección de prompts) en un documento del corpus y comprueba si el agente RAG obedece la orden escondida. Propón una defensa.
6. Cambia `PRINCIPAL` a otro proveedor y repite las demostraciones: ¿cuáles se comportan distinto?
7. Repite la comparación de agentes con un modelo local mayor (por ejemplo `qwen3:14b` o `gpt-oss:20b`, si los tienes en Ollama) y compara con los 2 de 6 del modelo pequeño.

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Yao, S. et al. (2023). ReAct: Synergizing Reasoning and Acting in Language Models. *ICLR*.
- Schick, T. et al. (2023). Toolformer: Language Models Can Teach Themselves to Use Tools. *NeurIPS*.
- Lewis, P. et al. (2020). Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks. *NeurIPS*.
- Anthropic (2024). Building effective agents. [anthropic.com/engineering/building-effective-agents](https://www.anthropic.com/engineering/building-effective-agents)
- LangChain. Documentación de LangChain y LangGraph. [docs.langchain.com](https://docs.langchain.com)
- LangChain. Código fuente de LangChain y LangGraph. [github.com/langchain-ai](https://github.com/langchain-ai)